# Search-03f · Réparer localement sous garantie : la recherche incrémentale LPA*

Ce carnet approfondit [Search-03 (recherche informée et A*)](Search-03-Informed.ipynb) : A\* calcule un plan optimal **une fois pour toutes** — mais le monde, lui, bouge. Une route se bloque, une cellule se révèle infranchissable, et le plan optimal d'hier traverse un mur. La question de ce carnet est celle de l'**opération 6** :

> **Réparer localement sous garantie** — réparer un plan après un changement *local*, en ne recalculant que ce que le changement a invalidé, tout en **transportant la garantie** (ici : l'optimalité) du calcul initial.

C'est la même opération que le *safe subgame solving* attesté côté GameTheory par le binôme **GT-13b/13c** : réparer la stratégie dans le sous-jeu affecté, la garantie globale reste tenue. Ce carnet en est la **seconde attestation**, sur un autre substrat : la recherche de chemin incrémentale.

**Les trois personnages :**

| Stratégie | Ce qu'elle fait | Garantie |
|---|---|---|
| **A\* from scratch** | oublie tout, recalcule tout | optimale, mais payée 273 expansions |
| **Repair naïf** | suit le plan initial, contourne l'obstacle, rejoint le plan | aucune — c'est le contre-témoin |
| **LPA\*** (Koenig & Likhachev, 2002) | réutilise les estimations $g$ encore valides, ne répare que les incohérences | **optimale, transportée** — payée 12 expansions |

**Ce que le carnet atteste, en une phrase :** la garantie d'optimalité n'est pas une propriété du calcul complet, mais de la **propagation des incohérences** — couper la propagation produit un plan invalide ou un coût faux ; la réparer entièrement rend l'optimalité *au prix uniquement de ce qui a changé*.

**Palier « mémoire bornée » (§7 à §11).** La première moitié de ce carnet comptait le travail ; la seconde mesure l'autre prix d'une garantie — la **mémoire**. A\*, IDA\* et A\* à frontière plafonnée sont confrontés sur les mêmes instances du taquin 8, jusqu'à ce qu'un plafond trop bas fasse perdre à l'algorithme son optimalité — silencieusement (exercices 7 à 9).

**Navigation** : [<< Search-03e — optimalité de A\*](Search-03e-AStar-Optimality.ipynb) | [Index](README.md) | [retour au parent Search-03 (A\*) ↑](Search-03-Informed.ipynb) | [jumeau Python](Search-03f-Reparer-Localement-Sous-Garantie-Python.ipynb)

In [1]:
// Le monde : grille 24x16, 4-connexe, couts unitaires. Un couloir muré force le passage.
using System;
using System.Collections.Generic;
using System.Linq;
using System.Globalization;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

const double INF = double.PositiveInfinity;

class Grid
{
    public int w, h;
    public HashSet<(int x, int y)> walls;
    public Grid(int w, int h, IEnumerable<(int x, int y)> walls)
    {
        this.w = w; this.h = h;
        this.walls = new HashSet<(int x, int y)>(walls);
    }
    public bool in_bounds((int x, int y) s) => s.x >= 0 && s.x < w && s.y >= 0 && s.y < h;
    public bool passable((int x, int y) s) => !walls.Contains(s);
    public IEnumerable<(int x, int y)> neighbors((int x, int y) s)
    {
        foreach (var d in new (int x, int y)[] { (1, 0), (-1, 0), (0, 1), (0, -1) })
        {
            var n = (s.x + d.x, s.y + d.y);
            if (in_bounds(n) && passable(n)) yield return n;
        }
    }
    public double cost((int x, int y) s, (int x, int y) t) => neighbors(s).Contains(t) ? 1 : double.PositiveInfinity;
}

int manhattan((int x, int y) a, (int x, int y) b) => Math.Abs(a.x - b.x) + Math.Abs(a.y - b.y);

(double cout, List<(int x, int y)> chemin, int expansions) astar(Grid grid, (int x, int y) start, (int x, int y) goal)
{
    // A* from scratch : tas + heuristique Manhattan. expansions[0] compte les noeuds depiles.
    var g = new Dictionary<(int x, int y), int> { [start] = 0 };
    var came = new Dictionary<(int x, int y), (int x, int y)>();
    var pq = new PriorityQueue<(int x, int y), (int f, long ordre)>();
    long tie = 0;
    int expansions = 0;
    pq.Enqueue(start, (manhattan(start, goal), tie++));
    while (pq.Count > 0)
    {
        var cur = pq.Dequeue();
        if (cur == goal)
        {
            var path = new List<(int x, int y)> { cur };
            while (came.ContainsKey(cur))
            {
                cur = came[cur];
                path.Add(cur);
            }
            path.Reverse();
            return (g[goal], path, expansions);
        }
        expansions++;
        foreach (var n in grid.neighbors(cur))
        {
            int ng = g[cur] + 1;
            if (!g.TryGetValue(n, out int old) || ng < old)
            {
                g[n] = ng;
                came[n] = cur;
                pq.Enqueue(n, (ng + manhattan(n, goal), tie++));
            }
        }
    }
    return (double.PositiveInfinity, new List<(int x, int y)>(), expansions);
}

void Exiger(bool condition, string message)
{
    if (!condition) throw new InvalidOperationException(message);
}

int W = 24, H = 16;
var BASE_WALLS = new HashSet<(int x, int y)>(Enumerable.Range(3, 16).Select(x => (x, 7)));
BASE_WALLS.UnionWith(new (int x, int y)[] { (10, 3), (11, 4), (12, 5), (6, 12), (7, 12), (8, 12) });
var START = (x: 1, y: 1);
var GOAL = (x: 22, y: 14);
var g_init = new Grid(W, H, BASE_WALLS);
var resultat_init = astar(g_init, START, GOAL);
double c_init = resultat_init.cout;
var plan_init = resultat_init.chemin;
int exp_init = resultat_init.expansions;
Exiger(c_init == 34 && plan_init.Count == 35 && exp_init == 285, "Parité du plan initial");
Exiger(c_init == manhattan(START, GOAL), "Le plan initial doit être monotone");
Console.WriteLine($"Plan initial : cout {c_init}, {plan_init.Count} cellules, {exp_init} expansions A*");
Console.WriteLine($"Un chemin monotone existe (cout = distance Manhattan {manhattan(START, GOAL)}) : {(c_init == manhattan(START, GOAL) ? "oui" : "non")}");

Plan initial : cout 34, 35 cellules, 285 expansions A*


Un chemin monotone existe (cout = distance Manhattan 34) : oui


### Lecture du résultat

Le plan initial est **monotone** : son coût (34) égale exactement la distance Manhattan — le mur du couloir ne force aucun détour, il y a un passage aligné. Le compteur d'expansions (le nombre de nœuds dépilés par A\*) va devenir notre monnaie : c'est lui qui mesure *ce qu'un calcul a dû apprendre*.

Le détail chiffré : le plan a un sommet de plus que son coût — 34 mouvements, donc 35 sommets — et ce premier calcul a dépilé 285 nœuds. Ce chiffre de 285 expansions est la ligne de base : c'est contre lui que seront comparés les 273 expansions du recalcul après changement — presque le même travail pour un monde différent.

In [2]:
// Le monde change : une cellule du plan optimal se bloque (nouvel obstacle).
var blocked = plan_init[10];  // sur le plan optimal, assez tot pour forcer un vrai contournement
Exiger(!BASE_WALLS.Contains(blocked) && blocked != START && blocked != GOAL, "Blocage invalide");
var murs_apres = new HashSet<(int x, int y)>(BASE_WALLS) { blocked };
var g_apres = new Grid(W, H, murs_apres);
var resultat_scratch = astar(g_apres, START, GOAL);
double c_scratch = resultat_scratch.cout;
var plan_scratch = resultat_scratch.chemin;
int exp_scratch = resultat_scratch.expansions;
Exiger(blocked == (11, 1) && c_scratch == 34 && exp_scratch == 273, "Parité du recalcul");
Console.WriteLine($"Cellule bloquee : {blocked} (etait sur le plan optimal)");
Console.WriteLine($"A* from scratch : cout {c_scratch}, {exp_scratch} expansions");

Cellule bloquee : (11, 1) (etait sur le plan optimal)


A* from scratch : cout 34, 273 expansions


### Lecture du résultat

Le nouveau plan optimal coûte toujours 34 — la grille offrait un second chemin monotone. Mais pour le découvrir, A\* **a tout réappris** : 273 expansions, comme si aucun calcul n'avait jamais eu lieu. C'est le coût de l'amnésie : le changement a invalidé *une cellule*, et la refacturation porte sur *toute la grille*.

La cellule bloquée est `(11, 1)`, choisie sur le plan optimal : le changement vise exactement là où le calcul précédent avait investi. Détail qui rend le constat plus net : 273 expansions après le changement contre 285 pour le calcul initial — le recalcul de zéro ne « refait pas plus » quand le changement est petit, il refait *tout*, quelle que soit l'ampleur du changement.

In [3]:
// Contre-temoin : le repair nave (suivre le plan, contourner, rejoindre).
(double cout, List<(int x, int y)> chemin, int expansions) naive_repair(Grid grid, List<(int x, int y)> plan, (int x, int y) blocked_cell)
{
    // suit le plan jusqu'a la cellule bloquee, contourne par le plus court chemin LOCAL
    // vers le premier waypoint du plan situe au-dela de l'obstacle, puis rejoint le plan tel quel.
    // retourne (cout total, plan, expansions du detour local).
    int i = plan.IndexOf(blocked_cell);
    if (i <= 0) throw new ArgumentException("Blocage absent du plan ou placé au départ");
    var before = plan.Take(i).ToList();
    var after = plan.Skip(i).ToList();
    int rejoin_idx = -1;
    for (int j = 1; j < after.Count; j++)
    {
        if (grid.passable(after[j])) { rejoin_idx = j; break; }
    }
    if (rejoin_idx < 0) return (double.PositiveInfinity, new List<(int x, int y)>(), 0);
    var detour = astar(grid, before[before.Count - 1], after[rejoin_idx]);
    if (double.IsPositiveInfinity(detour.cout)) return (double.PositiveInfinity, new List<(int x, int y)>(), detour.expansions);
    double total = before.Count - 1 + detour.cout + after.Count - 1 - rejoin_idx;
    var chemin = before.Take(before.Count - 1).Concat(detour.chemin).Concat(after.Skip(rejoin_idx + 1)).ToList();
    return (total, chemin, detour.expansions);
}

var resultat_naif = naive_repair(g_apres, plan_init, blocked);
double c_naif = resultat_naif.cout;
var plan_naif = resultat_naif.chemin;
int exp_naif = resultat_naif.expansions;
Exiger(c_naif == 36 && exp_naif == 8 && c_naif - c_scratch == 2, "Parité du repair naïf");
Exiger(plan_naif.First() == START && plan_naif.Last() == GOAL && plan_naif.Count - 1 == c_naif, "Extrémités ou longueur du chemin naïf");
Exiger(plan_naif.All(g_apres.passable) && Enumerable.Range(0, plan_naif.Count - 1).All(i => g_apres.neighbors(plan_naif[i]).Contains(plan_naif[i + 1])), "Chemin naïf invalide");
Console.WriteLine($"Repair nave : cout {c_naif} (detour local : {exp_naif} expansions)  |  optimal reel : {c_scratch}");
Console.WriteLine($"Dette du repair nave : +{c_naif - c_scratch} (chemin valide, optimalite perdue)");
Console.WriteLine("Le plan naive rejoint le plan initial et le suit tel quel apres le detour : rien ne l'incite a reviser une decision devenue sous-optimale");

Repair nave : cout 36 (detour local : 8 expansions)  |  optimal reel : 34


Dette du repair nave : +2 (chemin valide, optimalite perdue)


Le plan naive rejoint le plan initial et le suit tel quel apres le detour : rien ne l'incite a reviser une decision devenue sous-optimale


### Lecture du résultat

Le repair naïf produit un chemin **valide** — on arrive bien au but — mais **sans garantie** : +2 de dette par rapport à l'optimum. C'est le témoin exigé par l'opération 6 : *la déviation est mesurable quand la loi est violée*. La loi dit « conditions de bord préservant la garantie » ; le naïf n'a pas de conditions de bord du tout : il répare le point de friction et déclare le reste du plan intouchable, alors que l'apparition de l'obstacle a pu déplacer l'optimum **ailleurs**.

Le détour local ne coûte que 8 expansions : c'est précisément ce qui rend le contre-témoin dangereux — un chemin valide à +2, obtenu presque gratuitement, qu'aucun signal d'alerte n'accompagne.

## LPA\* : garder la mémoire, réparer l'incohérence

LPA\* (Lifelong Planning A\*, Koenig & Likhachev 2002) reprend exactement les quantités d'A\* et en ajoute une :

- $g(s)$ : le coût actuel du meilleur chemin trouvé vers $s$ — **l'estimation héritée** du calcul précédent ;
- $rhs(s)$ (« right-hand side ») : ce que ce coût **devrait être**, recalculé localement : $rhs(s) = \min_{s' \to s}\big(g(s') + c(s', s)\big)$, et $rhs(start) = 0$ ;
- un sommet est **localement consistant** si $g(s) = rhs(s)$. L'incohérence $g \neq rhs$ signifie : *ce que je crois et ce que je devrais croire diffèrent* — pile là où le changement a touché.

La repair tourne autour d'une **file de priorité U** d'incohérences, triée par la clé $k(s) = \big(\min(g, rhs) + h(s),\ \min(g, rhs)\big)$. Dépiler une incohérence $g > rhs$ **relève** $g$ (on a trouvé mieux) ; $g < rhs$ **abaisse** $g$ à l'infini (ce qu'on croyait n'existe plus) et replace les voisins dans la file. L'algorithme s'arrête dès que la clé minimale dépasse celle du but **et** que le but est consistant : tout ce qui n'a pas été touché n'a jamais été re-dépilé. La garantie d'optimalité d'A\* est transportée au repair ; seule la propagation paie.

In [4]:
// Implementation LPA* : g, rhs, file d'incoherences U, cles k = (k1, k2).
class LPAStar
{
    public Grid grid;
    public (int x, int y) start, goal;
    public Dictionary<(int x, int y), double> g = new Dictionary<(int x, int y), double>();
    public Dictionary<(int x, int y), double> rhs = new Dictionary<(int x, int y), double>();
    public Dictionary<(int x, int y), (double k1, double k2, long ordre)> U = new Dictionary<(int x, int y), (double k1, double k2, long ordre)>();
    public int expansions = 0;
    private long ordre = 0;
    private const double Infini = double.PositiveInfinity;

    public LPAStar(Grid grid, (int x, int y) start, (int x, int y) goal)
    {
        this.grid = grid; this.start = start; this.goal = goal;
        rhs[start] = 0;
        _insert(start, _calc_key(start));
    }
    public double Valeur(Dictionary<(int x, int y), double> d, (int x, int y) s) => d.TryGetValue(s, out var v) ? v : Infini;
    public (double k1, double k2) _calc_key((int x, int y) s)
    {
        double m = Math.Min(Valeur(g, s), Valeur(rhs, s));
        return (m + Math.Abs(s.x - goal.x) + Math.Abs(s.y - goal.y), m);
    }
    public void _insert((int x, int y) s, (double k1, double k2) k) => U[s] = (k.k1, k.k2, ordre++);
    public ((int x, int y) sommet, (double k1, double k2) cle) _top()
    {
        // sommet d'incoherence de plus petite cle (k1, k2) lexicographique
        if (U.Count == 0) return (default((int x, int y)), (Infini, Infini));
        var premier = U.OrderBy(e => e.Value.k1).ThenBy(e => e.Value.k2).ThenBy(e => e.Value.ordre).First();
        return (premier.Key, (premier.Value.k1, premier.Value.k2));
    }
    public void _update_vertex((int x, int y) u)
    {
        // rapproche rhs(u) de la realite locale, puis (re)insere u dans U s'il reste incoherent
        if (u != start)
            rhs[u] = grid.neighbors(u).Select(s => Valeur(g, s) + grid.cost(s, u)).DefaultIfEmpty(Infini).Min();
        U.Remove(u);
        if (Valeur(g, u) != Valeur(rhs, u)) _insert(u, _calc_key(u));
    }
    public double compute_shortest_path()
    {
        while (true)
        {
            var top = _top();
            var cleBut = _calc_key(goal);
            bool cleInferieure = top.cle.k1 < cleBut.k1 || (top.cle.k1 == cleBut.k1 && top.cle.k2 < cleBut.k2);
            if (!cleInferieure && Valeur(g, goal) == Valeur(rhs, goal)) break;
            if (U.Count == 0) throw new InvalidOperationException("File vide avec but incohérent");
            var s_top = top.sommet;
            expansions++;
            U.Remove(s_top);
            if (Valeur(g, s_top) > Valeur(rhs, s_top))
            {
                g[s_top] = rhs[s_top]; // raise : on a trouve mieux
                foreach (var s in grid.neighbors(s_top)) _update_vertex(s);
            }
            else
            {
                g[s_top] = Infini; // lower : ce qu'on croyait n'existe plus
                foreach (var s in grid.neighbors(s_top).Concat(new[] { s_top })) _update_vertex(s);
            }
        }
        return Valeur(g, goal);
    }
    public void change_walls(IEnumerable<(int x, int y)> added = null, IEnumerable<(int x, int y)> removed = null)
    {
        // applique le changement puis declare incoherents les sommets dont le voisinage a bouge
        var ajouts = (added ?? Enumerable.Empty<(int x, int y)>()).ToList();
        var retraits = (removed ?? Enumerable.Empty<(int x, int y)>()).ToList();
        foreach (var c in ajouts) grid.walls.Add(c);
        foreach (var c in retraits) grid.walls.Remove(c);
        foreach (var c in ajouts.Concat(retraits).Distinct())
        {
            foreach (var s in grid.neighbors(c)) _update_vertex(s);
            _update_vertex(c);
        }
    }
    public List<(int x, int y)> path()
    {
        if (double.IsPositiveInfinity(Valeur(g, goal))) return new List<(int x, int y)>();
        var chemin = new List<(int x, int y)> { goal };
        var cur = goal;
        while (cur != start)
        {
            var actuel = cur;
            cur = grid.neighbors(actuel).OrderBy(s => Valeur(g, s) + grid.cost(s, actuel)).First();
            chemin.Add(cur);
        }
        chemin.Reverse();
        return chemin;
    }
}

// sanite : premier calcul complet, doit reproduire A* exactement.
// le plan optimal n'est PAS unique (plusieurs chemins de meme cout) : on compare
// le cout et la longueur, jamais l'identite des cellules.
var lp_check = new LPAStar(new Grid(W, H, BASE_WALLS), START, GOAL);
double c_check = lp_check.compute_shortest_path();
Exiger(c_check == c_init && lp_check.path().Count == plan_init.Count && lp_check.expansions == 286, "Parité du premier passage LPA*");
Console.WriteLine($"Sanite : premier passage LPA* = A* (cout {c_init}, meme longueur de plan, {lp_check.expansions} expansions)");

Sanite : premier passage LPA* = A* (cout 34, meme longueur de plan, 286 expansions)


### Lecture du résultat

Le premier passage de LPA\* est volontairement équivalent à A\* (même coût, même plan) : c'est le **contrôle de cohérence** — le repair ne peut pas mieux faire qu'un calcul complet quand il part de zéro. Toute la valeur de LPA\* est dans les passages **suivants**.

Chiffré, ce premier passage a coûté 286 expansions, contre 285 pour le A\* direct — l'équivalent à un nœud près, ce que le contrôle de cohérence doit montrer : LPA\* partant de zéro ne fait ni mieux ni moins bien qu'un calcul complet.

In [5]:
// Le repair : le MEME changement de monde, traite par LPA* sans rien oublier.
var lp = new LPAStar(new Grid(W, H, BASE_WALLS), START, GOAL);
lp.compute_shortest_path();
int e_avant = lp.expansions;
lp.change_walls(added: new[] { blocked }); // declaration de l'incoherence, pas de re-calcul
double c_lpa = lp.compute_shortest_path(); // propagation minimale
int e_repair = lp.expansions - e_avant;
Exiger(c_lpa == c_scratch && c_lpa == 34 && e_repair == 12 && exp_scratch == 273, "Parité du repair LPA*");
Console.WriteLine($"LPA* repair : cout {c_lpa} (={(c_lpa == c_scratch ? c_scratch.ToString(CultureInfo.InvariantCulture) + " = optimal" : "ECART")})");
Console.WriteLine($"Expansions du repair : {e_repair}  |  A* from scratch : {exp_scratch}");
Console.WriteLine($"Reutilisation : le repair coute {(double)exp_scratch / e_repair:F0}x moins cher que tout reapprendre");

LPA* repair : cout 34 (=34 = optimal)


Expansions du repair : 12  |  A* from scratch : 273


Reutilisation : le repair coute 23x moins cher que tout reapprendre


### Lecture du résultat

Les deux faces de l'opération 6, mesurées :

- **la garantie est transportée** : coût LPA\* = coût A\* from scratch, à l'égalité près — le repair ne dégrade jamais l'optimalité ;
- **la réparation est locale** : 12 expansions contre 273, un rapport de 23×. Tout ce que le changement n'a pas invalidé — la majorité de la grille — n'a jamais été re-dépilé.

Le repair n'est pas « un A\* bon marché » : c'est un **autre contrat**. A\* répond à « quel est le plan optimal dans ce monde ? » ; LPA\* répond à « mon plan reste-t-il optimal, et si non, qu'est-ce que le changement a détruit ? ».

## La garantie, écrite noir sur blanc

Pourquoi l'optimalité survit-elle au repair ? Parce que la sortie de `compute_shortest_path` n'est pas « un chemin », c'est un **état cohérent** : à l'arrêt, $g(goal) = rhs(goal)$ et la clé minimale de U dépasse celle du but — ce qui signifie (théorème 3 de l'article) que $g$ coïncide avec les coûts de plus court chemin **partout où cela importe pour extraire le plan**. Les sommets non touchés n'ont pas changé de $g$ parce que rien ne pouvait les changer ; les sommets touchés ont été propagés jusqu'à cohérence. Il n'existe pas d'état intermédiaire « à peu près cohérent » qui s'arrête tôt : **la clause d'arrêt est la condition de bord de la loi**.

La correspondance terme à terme avec le safe subgame solving (GT-13b/13c) :

| Safe subgame solving (jeux) | LPA\* (chemins) |
|---|---|
| réparer la stratégie dans le sous-jeu atteint | réparer $g$ le long des incohérences |
| la garantie de non-exploitabilité reste tenue hors du sous-jeu | l'optimalité reste tenue hors de la zone touchée |
| contre-témoin : raffinement naïf → exploitabilité mesurable | contre-témoin : repair naïf → dette +2 mesurée |
| imbricable : les repairs s'enchaînent | imbricable : chaque changement repart de l'état cohérent précédent |

C'est la **seconde attestation indépendante** de l'opération 6 : deux substrats (stratégies de jeu, chemins), deux moteurs, même patron — réparer localement, garantir globalement.

In [6]:
// Imbricabilite : une SEQUENCE de 8 changements (murs ajoutes puis retires), seed fixe.
// CITE : changements tirés par random.seed(7) dans le jumeau Python, repris comme données d'entrée.
var changements_cites = new (bool ajout, (int x, int y) cellule)[]
{
    (true, (12, 13)), (true, (6, 2)), (false, (12, 13)), (true, (2, 6)),
    (true, (3, 5)), (false, (6, 2)), (true, (4, 3)), (true, (14, 10))
};
var expansions_citees = new[] { 1, 1, 1, 1, 2, 1, 1, 1 };
var g_seq = new Grid(W, H, BASE_WALLS);
var lp_seq = new LPAStar(g_seq, START, GOAL);
lp_seq.compute_shortest_path();
int concordances = 0, exp_lpa_total = 0, exp_scratch_total = 0;
for (int t = 0; t < changements_cites.Length; t++)
{
    var changement = changements_cites[t];
    if (!changement.ajout)
    {
        // un mur part
        Exiger(g_seq.walls.Contains(changement.cellule) && !BASE_WALLS.Contains(changement.cellule), "Retrait de mur invalide");
        lp_seq.change_walls(removed: new[] { changement.cellule });
    }
    else
    {
        // ne bloque pas le plan courant
        Exiger(g_seq.in_bounds(changement.cellule) && g_seq.passable(changement.cellule) && changement.cellule != START && changement.cellule != GOAL && !lp_seq.path().Contains(changement.cellule), "Ajout de mur invalide");
        lp_seq.change_walls(added: new[] { changement.cellule });
    }
    int e0 = lp_seq.expansions;
    double c_l = lp_seq.compute_shortest_path();
    int e_repair_seq = lp_seq.expansions - e0;
    exp_lpa_total += e_repair_seq;
    var reference = astar(new Grid(W, H, g_seq.walls), START, GOAL);
    exp_scratch_total += reference.expansions;
    bool ok = c_l == reference.cout;
    if (ok) concordances++;
    Exiger(ok && c_l == 34 && e_repair_seq == expansions_citees[t], "Parité d'une étape de la séquence");
    Exiger(lp_seq.path().Count == reference.chemin.Count, "Longueur du chemin après repair");
    string signe = changement.ajout ? "+" : "-";
    Console.WriteLine($"t={t} murs {signe}[{changement.cellule}] : LPA* {c_l} vs scratch {reference.cout} {(ok ? "OK" : "MISMATCH")} (repair {e_repair_seq} expansions)");
}
Exiger(concordances == 8 && exp_lpa_total == 9 && exp_scratch_total == 2258, "Parité du cumul de la séquence");
Console.WriteLine($"\nConcordances : {concordances}/8 | expansions cumulees : LPA* {exp_lpa_total} vs scratch {exp_scratch_total}");

t=0 murs +[(12, 13)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=1 murs +[(6, 2)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=2 murs -[(12, 13)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=3 murs +[(2, 6)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=4 murs +[(3, 5)] : LPA* 34 vs scratch 34 OK (repair 2 expansions)


t=5 murs -[(6, 2)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=6 murs +[(4, 3)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=7 murs +[(14, 10)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)



Concordances : 8/8 | expansions cumulees : LPA* 9 vs scratch 2258


### Lecture du résultat

Huit changements, huit concordances exactes avec la référence recalculée de zéro — l'optimalité est transportée **à chaque étape**, et chaque repair repart de l'état cohérent précédent (imbricabilité). Le budget cumulé de LPA\* reste une fraction de celui de la référence : c'est le sens de *lifelong* dans le nom — sur la durée de vie d'un problème qui change, l'amortissement du calcul initial se cumule.

Le détail par étape : les huit repairs coûtent respectivement 1, 1, 1, 1, 2, 1, 1 et 1 expansions — un seul changement (t=4, ajout de `(3, 5)`) a propagé un peu plus loin. Au total, 9 expansions cumulées pour LPA\* contre 2258 pour la référence recalculée de zéro à chaque fois : l'amortissement se cumule sur toute la durée de vie du problème, et la concordance 8/8 porte sur le coût comme sur la longueur du plan.

In [7]:
// Recapitulatif des trois strategies sur le changement unique.
var rows = new (string nom, double cout, string garantie, int expansions)[]
{
    ("A* from scratch", c_scratch, "optimale", exp_scratch),
    ("Repair nave", c_naif, $"aucune (dette +{c_naif - c_scratch})", exp_naif),
    ("LPA* repair", c_lpa, "optimale (transportee)", e_repair)
};
Exiger(rows[0].cout == 34 && rows[0].expansions == 273 && rows[1].cout == 36 && rows[1].expansions == 8 && rows[2].cout == 34 && rows[2].expansions == 12, "Parité du récapitulatif");
Console.WriteLine($"{"Strategie",-18} {"Cout",5}  {"Garantie",-26} {"Expansions",10}");
foreach (var row in rows)
    Console.WriteLine($"{row.nom,-18} {row.cout,5}  {row.garantie,-26} {row.expansions,10}");
Console.WriteLine();
Console.WriteLine("Le repair nave paie peu (detour local) mais ne garantit rien ;");
Console.WriteLine("LPA* garantit l'optimalite au prix exactement de ce que le changement a invalide.");

Strategie           Cout  Garantie                   Expansions


A* from scratch       34  optimale                          273


Repair nave           36  aucune (dette +2)                   8


LPA* repair           34  optimale (transportee)             12


Le repair nave paie peu (detour local) mais ne garantit rien ;


LPA* garantit l'optimalite au prix exactement de ce que le changement a invalide.


### Lecture du résultat, et la dette restante

Le tableau est le résumé de la loi : *aucune stratégie n'a les trois colonnes à la fois gratuites*. Le naïf économise le calcul et perd la garantie ; le scratch a la garantie et paie tout ; LPA\* a la garantie et ne paie que le delta. **La dette reste ouverte** sur deux points, à traiter ailleurs :

- LPA\* suppose le monde **connu** : quand l'obstacle n'est observé qu'à proximité, il faut D\* Lite (LPA\* + déplacement de l'agent, but et départ échangés) — piste naturelle pour un approfondissement ultérieur ;
- les coûts unitaires cachent le cas pondéré (terrain à $c(s, s')$ variable) — le mécanisme de `rhs` le supporte tel quel, mais les témoins seraient à re-mesurer.

In [8]:
// Test negatif de l'op 6 : couper la propagation = croire une valeur fausse.
// Monde a goulet : mur vertical x=12, ouverture unique en (12, 8), bords haut/bas ouverts.
int W2 = 24, H2 = 16;
var PONT = (x: 12, y: 8);
var mur_goulet = new HashSet<(int x, int y)>(Enumerable.Range(1, H2 - 2).Select(y => (12, y)));
mur_goulet.Remove(PONT);
var g_pont = new Grid(W2, H2, mur_goulet);
var resultat_pont = astar(g_pont, START, GOAL);
double c_avant = resultat_pont.cout;
var plan_pont = resultat_pont.chemin;
Exiger(plan_pont.Contains(PONT) && c_avant == 34, "Plan initial du goulet");
Console.WriteLine($"Goulet : plan initial cout {c_avant}, unique passage en {PONT}");

// (a) AUCUNE propagation : on meure le pont, l'etat interne ne bouge pas
var lp_croire = new LPAStar(new Grid(W2, H2, mur_goulet), START, GOAL);
lp_croire.compute_shortest_path();
lp_croire.grid.walls.Add(PONT); // le monde change, l'etat interne non
double c_cru = lp_croire.g[GOAL];
Console.WriteLine($"(a) sans propagation : g(goal) affirme toujours {c_cru}");

// (b) reference honnete : A* from scratch apres blocage du pont
var murs_pont_ferme = new HashSet<(int x, int y)>(mur_goulet) { PONT };
var resultat_goulet = astar(new Grid(W2, H2, murs_pont_ferme), START, GOAL);
double c_vrai = resultat_goulet.cout;
int exp_goulet = resultat_goulet.expansions;
Console.WriteLine($"(b) optimum reel apres blocage : {c_vrai} ({exp_goulet} expansions scratch)");

// (c) propagation COMPLETE par LPA* : la meme instance, traitee honnetement
var lp_vrai = new LPAStar(new Grid(W2, H2, mur_goulet), START, GOAL);
lp_vrai.compute_shortest_path();
int e_avant_g = lp_vrai.expansions;
lp_vrai.change_walls(added: new[] { PONT });
double c_lpa_g = lp_vrai.compute_shortest_path();
int e_repair_g = lp_vrai.expansions - e_avant_g;
Exiger(c_cru == 34 && c_vrai == 36 && exp_goulet == 276 && c_lpa_g == c_vrai && e_repair_g == 269, "Parité du test négatif");
Console.WriteLine($"(c) LPA* repair : {c_lpa_g} en {e_repair_g} expansions (vs {exp_goulet} au scratch -- un changement structurel n'economise presque rien)");
Console.WriteLine($"Ecart de croyance : {c_cru} affirme vs {c_vrai} reel -> la garantie n'etait pas deposee dans le calcul initial, elle est retablie par la propagation");

Goulet : plan initial cout 34, unique passage en (12, 8)


(a) sans propagation : g(goal) affirme toujours 34


(b) optimum reel apres blocage : 36 (276 expansions scratch)


(c) LPA* repair : 36 en 269 expansions (vs 276 au scratch -- un changement structurel n'economise presque rien)


Ecart de croyance : 34 affirme vs 36 reel -> la garantie n'etait pas deposee dans le calcul initial, elle est retablie par la propagation


### Lecture du résultat

Deux enseignements, le second n'étant pas le plus confortable :

- **La croyance fausse est mesurée** : sans propagation, $g(goal)$ affirme 34 alors que l'optimum réel est 36. Un état interne non propagé n'est pas « un peu optimiste » : il **ment**. On n'en extrait même pas de plan — un état incohérent n'a pas de chemin bien défini. La garantie d'optimalité n'est donc pas une propriété que le premier calcul aurait « déposée » quelque part : elle est **ré-établie à chaque changement par la propagation des incohérences** — les conditions de bord (jusqu'où propager) ne sont pas un détail d'implémentation, elles SONT la garantie.
- **La localité paie quand le changement est local** : ici le repair LPA\* coûte presque autant que le scratch (269 contre 276), car bloquer l'unique passage invalide *tout* ce qui est au-delà du pont. LPA\* ne dégénère jamais en pire que scratch, mais son avantage (23× sur le changement local du début) disparaît quand le changement est structurel. C'est la dette honnête de l'opération : la garantie est transportée, l'économie ne l'est que si le monde change localement.

## Recoller : compatibilité puis composition

C'est le prolongement direct de la section précédente, par l'autre bout : LPA\* **répare un plan** quand le monde change ; cette section **recelle des tables locales** en un objet global. Les deux posent la même question — *qu'est-ce qu'un calcul local garantit au niveau global ?*

Cette section examine l'**opération 5**, le recollement d'objets locaux compatibles. Elle en propose une seconde attestation sur le substrat de la recherche de chemin : des fenêtres résolues localement doivent pouvoir être intégrées sans contradiction. Le parallèle avec l'exemple de de Finetti est le passage d'une incohérence locale à un **témoin exploitable** ; l'exemple Lean formalise ce cadre, tandis qu'ici le témoin est une marche concrète sur une grille. Aucun outil Lean n'est nécessaire pour les calculs de ce carnet.

Le décor évoque **HPA\***, où un planificateur hiérarchique organise une carte en régions et utilise des portails pour relier des sous-problèmes locaux. Ici, les fenêtres et leurs tables d'intégration constituent une instance pédagogique explicite : elles permettent de mesurer ce qui se passe lorsque les correspondances ne se composent pas en l'identité.

**Critère de la Tombée 3** ([ledger](../../../docs/transients/2026-08-24-12204-ict-chantier-1-audit-froid.md)) : le témoin exigé est **exploitable, pas le résidu** — « si nous prétendons détecter un défaut de recollement, pouvons-nous produire un cycle concret qui exploite ce défaut ? » Cette section produit ce cycle : une **marche de méidentification exécutée pas à pas** sur la grille, qui traverse son but puis le dépasse en croyant arriver. La dette d'ICT-15d est respectée : aucun mot de cadre abstrait n'est invoqué avant que les transports soient **possédés** — ici ce sont des permutations explicites, composées et mesurées.

*Posture : pas d'auto-promotion. La décision de promotion de l'op 5 vers la table appartient à la relecture de l'EPIC (même posture que la section 12 de [Search-03b](Search-03b-PatternDatabases.ipynb) pour l'op 3 et que GT-19 pour l'op 2).*

### La loi et le décor

**La loi de l'opération** : une famille de correspondances locales se recolle en un objet global cohérent si et seulement si (i) chaque **chevauchement** porte une correspondance compatible (ici : une bijection complète des états partagés) et (ii) la **composition autour de chaque chevauchement triple** est l'identité. La condition (i) est celle que tout intégrateur écrit ; la condition (ii) est celle que personne n'écrit — c'est précisément là que le défaut vit.

**Le décor, réaliste** : une carte est couverte par trois fenêtres rectangulaires chevauchantes (les clusters d'un planificateur hiérarchique). Chaque fenêtre résout **indépendamment** son sous-problème (BFS) et numérote **ses** états de bord (les portails) dans **son** ordre de découverte — trois intégrateurs, trois conventions. Entre chaque paire de fenêtres, une **table d'intégration** écrite à la main associe les numérotations locales. Le contrat vérifié à l'intégration est le contrat faible : chaque table est une bijection, la couverture est complète.

In [9]:
// Carte, fenetres (zones), murs et portails
int ROWS = 9, COLS = 11;
var WALLS = new HashSet<(int r, int c)> { (1, 2), (2, 5), (3, 5), (6, 4) };
// rectangle (r0, c0)-(r1, c1) inclus
var ZONES = new Dictionary<string, ((int r, int c) debut, (int r, int c) fin)>
{
    ["A"] = ((0, 0), (4, 6)),
    ["B"] = ((0, 4), (4, 10)),
    ["C"] = ((3, 1), (8, 7))
};
var REFS = new Dictionary<string, (int r, int c)> { ["A"] = (0, 0), ["B"] = (0, 10), ["C"] = (8, 1) }; // reference locale de chaque fenetre
// Portails : etats de bord partages par les TROIS fenetres (le triple chevauchement)
var PORTALS = new List<(int r, int c)> { (3, 4), (3, 6), (4, 4), (4, 6) };
bool inside(string z, int r, int c)
{
    var rectangle = ZONES[z];
    return rectangle.debut.r <= r && r <= rectangle.fin.r && rectangle.debut.c <= c && c <= rectangle.fin.c;
}
string zone_set(int r, int c) => string.Concat(new[] { "A", "B", "C" }.Where(z => inside(z, r, c)));
var zone_cells = new Dictionary<string, HashSet<(int r, int c)>>();
foreach (var z in new[] { "A", "B", "C" })
{
    var cellules = new HashSet<(int r, int c)>();
    for (int r = 0; r < ROWS; r++)
        for (int c = 0; c < COLS; c++)
            if (inside(z, r, c) && !WALLS.Contains((r, c))) cellules.Add((r, c));
    zone_cells[z] = cellules;
}
// Rendu : chaque position affiche la liste des fenetres qui la couvrent
Console.WriteLine("Couverture (mur = #) :");
for (int r = 0; r < ROWS; r++)
{
    var morceaux = new List<string>();
    for (int c = 0; c < COLS; c++)
        morceaux.Add(WALLS.Contains((r, c)) ? "  # " : zone_set(r, c).PadLeft(3) + " ");
    Console.WriteLine(string.Concat(morceaux));
}
Console.WriteLine();
Console.WriteLine("Portails du triple chevauchement :");
foreach (var p in PORTALS) Console.WriteLine($"  {p} -> couvert par {zone_set(p.r, p.c)}");
Exiger(PORTALS.All(p => zone_set(p.r, p.c) == "ABC"), "Portails hors du triple chevauchement");
// Le triple chevauchement complet, pour voir ou vit la composition
var triple = new List<(int r, int c)>();
for (int r = 0; r < ROWS; r++)
    for (int c = 0; c < COLS; c++)
        if (zone_set(r, c) == "ABC") triple.Add((r, c));
Exiger(triple.SequenceEqual(new (int, int)[] { (3, 4), (3, 5), (3, 6), (4, 4), (4, 5), (4, 6) }), "Triple chevauchement inattendu");
Console.WriteLine($"Triple chevauchement (A et B et C) : [{string.Join(", ", triple)}]");

Couverture (mur = #) :


  A   A   A   A  AB  AB  AB   B   B   B   B 


  A   A   #   A  AB  AB  AB   B   B   B   B 


  A   A   A   A  AB   #  AB   B   B   B   B 


  A  AC  AC  AC ABC   # ABC  BC   B   B   B 


  A  AC  AC  AC ABC ABC ABC  BC   B   B   B 


... (6 outputs left out)

  (3, 6) -> couvert par ABC


  (4, 4) -> couvert par ABC


  (4, 6) -> couvert par ABC


Triple chevauchement (A et B et C) : [(3, 4), (3, 5), (3, 6), (4, 4), (4, 5), (4, 6)]


### Lecture du décor

Les trois fenêtres se chevauchent **par paires** (bandes `AB`, `AC`, `BC`) et leur intersection commune forme le bloc `ABC` — c'est le **triple chevauchement**, le seul endroit où une correspondance peut être confrontée à elle-même en boucle. Les quatre portails vivent tous dans ce bloc : chaque paire de fenêtres les voit, aucun intégrateur ne les voit de la même façon.

La sortie précise le bloc commun : le triple chevauchement est exactement `[(3, 4), (3, 5), (3, 6), (4, 4), (4, 5), (4, 6)]` — six cases — et les quatre portails sont `(3, 4)`, `(3, 6)`, `(4, 4)`, `(4, 6)`. Tous vivent dans le bloc ABC : c'est le seul endroit où les trois numérotations peuvent être confrontées, et ce sera le siège de la violation mesurée plus loin.

In [10]:
// Sous-problemes locaux : BFS par fenetre depuis sa reference, numerotation par decouverte
(Dictionary<(int r, int c), int> dist, Dictionary<(int r, int c), (int r, int c)> parent) bfs(HashSet<(int r, int c)> cells, (int r, int c) src)
{
    var dist = new Dictionary<(int r, int c), int> { [src] = 0 };
    var parent = new Dictionary<(int r, int c), (int r, int c)>();
    var q = new Queue<(int r, int c)>();
    q.Enqueue(src);
    var directions = new (int r, int c)[] { (1, 0), (-1, 0), (0, 1), (0, -1) };
    while (q.Count > 0)
    {
        var courant = q.Dequeue();
        foreach (var d in directions)
        {
            var n = (r: courant.r + d.r, c: courant.c + d.c);
            if (cells.Contains(n) && !dist.ContainsKey(n))
            {
                dist[n] = dist[courant] + 1;
                parent[n] = courant;
                q.Enqueue(n);
            }
        }
    }
    return (dist, parent);
}
var numbering = new Dictionary<string, Dictionary<(int r, int c), int>>(); // fenetre -> {portail: indice local}
var by_num = new Dictionary<string, List<(int r, int c)>>(); // fenetre -> [portails par indice local]
var dists = new Dictionary<string, Dictionary<(int r, int c), Dictionary<(int r, int c), int>>>(); // fenetre -> {portail: {portail: distance locale}}
var parents = new Dictionary<string, Dictionary<(int r, int c), (int r, int c)>>();
foreach (var z in new[] { "A", "B", "C" })
{
    var resultat = bfs(zone_cells[z], REFS[z]);
    parents[z] = resultat.parent;
    Exiger(PORTALS.All(p => resultat.dist.ContainsKey(p)), $"Portail non atteignable dans {z}");
    var order = PORTALS.OrderBy(p => resultat.dist[p]).ToList(); // ordre de decouverte BFS
    numbering[z] = order.Select((p, i) => (p, i)).ToDictionary(v => v.p, v => v.i);
    by_num[z] = order;
    var table = new Dictionary<(int r, int c), Dictionary<(int r, int c), int>>();
    foreach (var p in PORTALS)
    {
        var dp = bfs(zone_cells[z], p).dist;
        table[p] = PORTALS.ToDictionary(q => q, q => dp[q]);
    }
    dists[z] = table;
    Console.WriteLine($"Fenetre {z} (reference {REFS[z]}) numerotation par decouverte :");
    for (int i = 0; i < order.Count; i++) Console.WriteLine($"   {z}#{i} = {order[i]}");
}
// Les trois conventions sont deux a deux distinctes : personne n'a coordonne
Exiger(!by_num["A"].SequenceEqual(by_num["B"]) && !by_num["B"].SequenceEqual(by_num["C"]) && !by_num["A"].SequenceEqual(by_num["C"]), "Numérotations identiques");
Console.WriteLine("Numerotations deux a deux distinctes : assert OK");
// Epingle : les ordres de decouverte attendus (garde la construction verifiable)
Exiger(by_num["A"].SequenceEqual(new (int, int)[] { (3, 4), (4, 4), (3, 6), (4, 6) }), "Ordre A");
Exiger(by_num["B"].SequenceEqual(new (int, int)[] { (3, 6), (4, 6), (3, 4), (4, 4) }), "Ordre B");
Exiger(by_num["C"].SequenceEqual(new (int, int)[] { (4, 4), (3, 4), (4, 6), (3, 6) }), "Ordre C");
Console.WriteLine("Ordres de decouverte epingles : assert OK");
var distancesA = new int[,] { { 0, 4, 1, 3 }, { 4, 0, 3, 1 }, { 1, 3, 0, 2 }, { 3, 1, 2, 0 } };
for (int i = 0; i < PORTALS.Count; i++)
    for (int j = 0; j < PORTALS.Count; j++)
        Exiger(dists["A"][PORTALS[i]][PORTALS[j]] == distancesA[i, j], "Distance locale A inattendue");
Console.WriteLine();
Console.WriteLine("Table locale de la fenetre A (distances entre portails via l'interieur de A) :");
foreach (var p in PORTALS)
    Console.WriteLine($"   {p} {{{string.Join(", ", PORTALS.Select(q => $"{q}: {dists["A"][p][q]}"))}}}");

Fenetre A (reference (0, 0)) numerotation par decouverte :


   A#0 = (3, 4)


   A#1 = (4, 4)


   A#2 = (3, 6)


   A#3 = (4, 6)


Fenetre B (reference (0, 10)) numerotation par decouverte :


... (12 outputs left out)

   (3, 4) {(3, 4): 0, (3, 6): 4, (4, 4): 1, (4, 6): 3}


   (3, 6) {(3, 4): 4, (3, 6): 0, (4, 4): 3, (4, 6): 1}


   (4, 4) {(3, 4): 1, (3, 6): 3, (4, 4): 0, (4, 6): 2}


   (4, 6) {(3, 4): 3, (3, 6): 1, (4, 4): 2, (4, 6): 0}


### Lecture des sous-problèmes

Chaque fenêtre a produit un objet local complet : une numérotation des portails (ordre de découverte depuis **sa** référence — coin haut-gauche pour A, coin haut-droit pour B, coin bas-gauche pour C) et une table de distances locales. Les trois numérotations diffèrent deux à deux : c'est le vécu de trois intégrations indépendantes, et la raison d'être des tables de correspondance.

Concrètement, le même portail physique porte des étiquettes différentes selon la fenêtre : `(3, 4)` vaut `A#0` pour A, mais `B#2` pour B et `C#1` pour C. La table locale de A, par exemple, rend la distance `(3, 4) -> (4, 4)` égale à 1 : c'est cette distance physique que la colle devra restituer, quelle que soit l'étiquette locale empruntée.

In [11]:
// Tables d'integration ecrites par paire (format reel : paires d'indices locaux)
// La table A->B porte l'erreur d'integration : les deux sorties (3,4) et (3,6)
// y sont CROISEES -- deux lignes voisines de la table, permutees a la main.
var pi_AB_idx = new Dictionary<int, int> { [0] = 0, [1] = 3, [2] = 2, [3] = 1 }; // A#i <-> B#j  (croisee sur la paire 0/2)
var pi_BC_idx = new Dictionary<int, int> { [0] = 3, [1] = 2, [2] = 1, [3] = 0 }; // correcte
var pi_CA_idx = new Dictionary<int, int> { [0] = 1, [1] = 0, [2] = 3, [3] = 2 }; // correcte

void show_binding(string name, string z1, string z2, Dictionary<int, int> binding)
{
    Console.WriteLine($"Table {name} ({z1} -> {z2}) :");
    foreach (var kv in binding.OrderBy(kv => kv.Key))
        Console.WriteLine($"   {z1}#{kv.Key} {by_num[z1][kv.Key]}  <->  {z2}#{kv.Value} {by_num[z2][kv.Value]}");
}
show_binding("pi_AB", "A", "B", pi_AB_idx);
show_binding("pi_BC", "B", "C", pi_BC_idx);
show_binding("pi_CA", "C", "A", pi_CA_idx);

// --- Le contrat faible, celui que l'integration verifie reellement ---
void check_pairwise(string name, Dictionary<int, int> binding)
{
    Exiger(binding.Values.OrderBy(x => x).SequenceEqual(Enumerable.Range(0, 4)), $"{name} n'est pas une bijection");
    Exiger(binding.Count == 4 && binding.Keys.OrderBy(x => x).SequenceEqual(Enumerable.Range(0, 4)), $"{name} ne couvre pas tout le chevauchement");
    Console.WriteLine($"{name} : bijection OK, couverture complete OK");
}
check_pairwise("pi_AB", pi_AB_idx);
check_pairwise("pi_BC", pi_BC_idx);
check_pairwise("pi_CA", pi_CA_idx);
Console.WriteLine();
Console.WriteLine("Compatibilite sur les chevauchements : VERIFIEE par le contrat faible (asserts OK)");

Table pi_AB (A -> B) :


   A#0 (3, 4)  <->  B#0 (3, 6)


   A#1 (4, 4)  <->  B#3 (4, 4)


   A#2 (3, 6)  <->  B#2 (3, 4)


   A#3 (4, 6)  <->  B#1 (4, 6)


Table pi_BC (B -> C) :


... (9 outputs left out)

pi_AB : bijection OK, couverture complete OK


pi_BC : bijection OK, couverture complete OK


pi_CA : bijection OK, couverture complete OK


Compatibilite sur les chevauchements : VERIFIEE par le contrat faible (asserts OK)


### Pourquoi la compatibilité par paires ne suffit pas

Le contrat faible ne consulte **aucune donnée croisée** : il vérifie que chaque table, prise seule, associe chaque état partagé de gauche à exactement un état partagé de droite. La table `pi_AB` croisée satisfait ce contrat parfaitement — une permutation en est une. Aucune vérification par paire ne compose les tables entre elles ; la contrainte qui tue n'est visible que lorsqu'on **compose autour du triple**.

Le cœur du défaut est lisible dans `pi_AB` : la table associe `A#0 (3, 4)` à `B#0 (3, 6)` — deux portails physiquement distincts — alors que `(3, 4)` porte chez B l'étiquette `B#2`. Chaque table prise seule est une bijection parfaite (les trois asserts « bijection OK, couverture complète OK » passent) ; c'est la correspondance croisée elle-même qui est fausse, et aucune vérification par paire ne peut le voir.

In [12]:
// Composition sur le triple : sigma = pi_AB o pi_BC o pi_CA (cellule -> cellule)
(int r, int c) convert((int r, int c) p, string z_from, Dictionary<int, int> binding, string z_to)
{
    int i = numbering[z_from][p];
    return by_num[z_to][binding[i]];
}
var sigma = new Dictionary<(int r, int c), (int r, int c)>();
foreach (var p in PORTALS)
{
    var p_b = convert(p, "A", pi_AB_idx, "B"); // A -> B
    var p_c = convert(p_b, "B", pi_BC_idx, "C"); // B -> C
    var p_a = convert(p_c, "C", pi_CA_idx, "A"); // C -> A (retour dans la numerotation A)
    sigma[p] = p_a;
}
var moved = PORTALS.Where(p => sigma[p] != p).ToList();
var fixes = PORTALS.Where(p => sigma[p] == p).ToList();
Console.WriteLine("sigma (aller-retour A -> B -> C -> A) :");
foreach (var p in PORTALS)
    Console.WriteLine($"   {p} -> {sigma[p]}   [{(sigma[p] == p ? "FIXE" : "DEPLACE")}]");
Console.WriteLine($"Points fixes : [{string.Join(", ", fixes)}]");
Console.WriteLine($"Points deplaces : [{string.Join(", ", moved)}]");

string cycle_notation(Dictionary<(int r, int c), (int r, int c)> s)
{
    var seen = new HashSet<(int r, int c)>();
    var cycles = new List<string>();
    foreach (var p in s.Keys.OrderBy(p => p.r).ThenBy(p => p.c))
    {
        if (seen.Contains(p)) continue;
        var cyc = new List<(int r, int c)>();
        var cur = p;
        while (seen.Add(cur)) { cyc.Add(cur); cur = s[cur]; }
        if (cyc.Count > 1) cycles.Add("(" + string.Join(" ", cyc) + ")");
    }
    return cycles.Count == 0 ? "identite" : string.Join(" ", cycles);
}
string notation_sigma = cycle_notation(sigma);
Console.WriteLine($"Notation cyclique de sigma : {notation_sigma}");
Exiger(moved.SequenceEqual(new (int, int)[] { (3, 4), (3, 6) }) && fixes.SequenceEqual(new (int, int)[] { (4, 4), (4, 6) }), "Points de sigma inattendus");
Exiger(sigma[(3, 4)] == (3, 6) && sigma[(3, 6)] == (3, 4) && notation_sigma == "((3, 4) (3, 6))", "Composition inattendue");
Console.WriteLine("Composition sur le triple : VIOLATION mesuree (sigma != identite)");

sigma (aller-retour A -> B -> C -> A) :


   (3, 4) -> (3, 6)   [DEPLACE]


   (3, 6) -> (3, 4)   [DEPLACE]


   (4, 4) -> (4, 4)   [FIXE]


   (4, 6) -> (4, 6)   [FIXE]


Points fixes : [(4, 4), (4, 6)]


Points deplaces : [(3, 4), (3, 6)]


Notation cyclique de sigma : ((3, 4) (3, 6))


Composition sur le triple : VIOLATION mesuree (sigma != identite)


### La loi violée

En composant les trois tables autour du triple chevauchement, un portail revient sur la position d'un **autre** : `(3,4)` et `(3,6)` sont échangés par l'aller-retour. Chaque table prise seule était irréprochable au contrat faible ; le produit, lui, n'est pas l'identité — la loi de l'opération 5 exige l'identité. La question de la Tombée 3 devient concrète : **ce défaut est-il exploitable ?**

La sortie chiffre la violation : sigma échange `(3, 4)` et `(3, 6)` (notation cyclique `((3, 4) (3, 6))`) et laisse fixes `(4, 4)` et `(4, 6)`. Sur les quatre portails du triple chevauchement, deux sont déplacés : le défaut n'est pas marginal, il touche la moitié d'entre eux.

In [13]:
// Non-recollabilite : AUCUNE numerotation globale n'est coherente avec les trois tables
// (enumeration exhaustive de toutes les etiquetages injectifs des portails)
IEnumerable<int[]> permutations(int n)
{
    var valeurs = new int[n];
    var utilise = new bool[n];
    IEnumerable<int[]> parcourir(int profondeur)
    {
        if (profondeur == n) { yield return (int[])valeurs.Clone(); yield break; }
        for (int i = 0; i < n; i++)
        {
            if (utilise[i]) continue;
            utilise[i] = true;
            valeurs[profondeur] = i;
            foreach (var perm in parcourir(profondeur + 1)) yield return perm;
            utilise[i] = false;
        }
    }
    return parcourir(0);
}
int count_global_numberings(Dictionary<int, int> b_ab, Dictionary<int, int> b_bc, Dictionary<int, int> b_ca)
{
    int ok_count = 0;
    var tables = new (string z1, string z2, Dictionary<int, int> b)[] { ("A", "B", b_ab), ("B", "C", b_bc), ("C", "A", b_ca) };
    foreach (var perm in permutations(PORTALS.Count))
    {
        var lab = PORTALS.Select((p, i) => (p, etiquette: perm[i])).ToDictionary(v => v.p, v => v.etiquette);
        bool ok = true;
        foreach (var table in tables)
            foreach (var kv in table.b)
                if (lab[by_num[table.z1][kv.Key]] != lab[by_num[table.z2][kv.Value]]) ok = false;
        if (ok) ok_count++;
    }
    return ok_count;
}
int n_ok = count_global_numberings(pi_AB_idx, pi_BC_idx, pi_CA_idx);
int total = permutations(PORTALS.Count).Count();
Exiger(n_ok == 0 && total == 24, "Énumération globale inattendue");
Console.WriteLine($"Numerotations globales coherentes : {n_ok} / {total} etiquetages injectifs testes");
Console.WriteLine("Argument en une ligne : toute etiquette coherente devrait satisfaire");
Console.WriteLine("lab[(3,4)] = lab[sigma[(3,4)]] = lab[(3,6)] -- deux portails distincts, meme etiquette :");
Console.WriteLine("l'injectivite est violee. Le recollement n'existe pas.");

Numerotations globales coherentes : 0 / 24 etiquetages injectifs testes


Argument en une ligne : toute etiquette coherente devrait satisfaire


lab[(3,4)] = lab[sigma[(3,4)]] = lab[(3,6)] -- deux portails distincts, meme etiquette :


l'injectivite est violee. Le recollement n'existe pas.


### Lecture du résultat

L'énumération exhaustive tranche : **0 numérotation globale cohérente sur les 24 étiquetages injectifs testés**. L'argument en une ligne de la sortie le montre directement : toute étiquette cohérente devrait satisfaire `lab[(3,4)] = lab[sigma[(3,4)]] = lab[(3,6)]` — deux portails distincts avec la même étiquette, l'injectivité est violée. Autrement dit, aucune numérotation globale des portails ne respecte les trois tables à la fois : le recollement n'existe pas — ce n'est pas un objet dégradé, c'est un objet qui n'est pas défini.

In [14]:
// Consequence mesuree : la table recollee repond DEUX valeurs a la meme requete physique
var requete = (depart: (r: 3, c: 4), arrivee: (r: 4, c: 4)); // deux portails voisins (une ligne l'un sous l'autre)
int reponse_directe = dists["A"][requete.depart][requete.arrivee];
// En passant par l'aller-retour tordu A -> B -> C -> A, la colle reidentifie le depart :
var depart_reidentifie = sigma[requete.depart];
int reponse_via_colle = dists["A"][depart_reidentifie][requete.arrivee];
Exiger(reponse_directe == 1 && reponse_via_colle == 3 && depart_reidentifie == (3, 6), "Réponses de la colle inattendues");
Console.WriteLine($"Requete physique : distance {requete.depart} -> {requete.arrivee}");
Console.WriteLine($"  reponse directe (table A)                    : {reponse_directe}");
Console.WriteLine($"  reponse apres reidentification par la colle  : {reponse_via_colle}");
Console.WriteLine($"     (la colle croit que le depart est {depart_reidentifie})");
Exiger(reponse_directe != reponse_via_colle, "Les deux réponses devraient diverger");
Console.WriteLine("Deux reponses pour une requete : l'objet recolle n'est pas une fonction.");

Requete physique : distance (3, 4) -> (4, 4)


  reponse directe (table A)                    : 1


  reponse apres reidentification par la colle  : 3


     (la colle croit que le depart est (3, 6))


Deux reponses pour une requete : l'objet recolle n'est pas une fonction.


### Lecture

La même paire physique de portails reçoit deux distances selon le chemin de consultation. Un objet recollé qui répond deux valeurs n'est pas une table dégradée : **il n'existe pas**. Toute couche au-dessus (un planificateur, un estimateur) consomme donc un objet incohérent — et le dommage devient observable par un adversaire.

La sortie précise le mécanisme : après réidentification par la colle, le départ `(3, 4)` est confondu avec `(3, 6)`, et la distance rendue pour la requête `(3, 4) -> (4, 4)` passe de 1 (réponse directe de la table A) à 3 (ce que la colle croit être la distance depuis `(3, 6)`).

In [15]:
// LE TEMOIN EXPLOITABLE (Loi I) : la marche de meidentification, executee pas a pas
// Situation : un agent est physiquement en (3,4) et demande a la colle un plan vers (4,4).
// La colle consulte son identification : depart (3,4) -> (apres aller-retour tordu) (3,6).
// Elle planifie donc dans la fenetre A depuis (3,6) et retourne les mouvements RELATIFS.
var start_vrai = (r: 3, c: 4);
var but = (r: 4, c: 4);
var depart_cru = sigma[start_vrai]; // ce que la colle croit etre le point de depart
List<(int r, int c)> path_from((int r, int c) src, (int r, int c) goal, string z)
{
    var par = bfs(zone_cells[z], src).parent;
    var chemin = new List<(int r, int c)> { goal };
    while (chemin[chemin.Count - 1] != src) chemin.Add(par[chemin[chemin.Count - 1]]);
    chemin.Reverse();
    return chemin;
}
var chemin_cru = path_from(depart_cru, but, "A"); // plan calcule depuis la position crue
var moves = new List<(int r, int c)>();
for (int i = 0; i < chemin_cru.Count - 1; i++)
    moves.Add((chemin_cru[i + 1].r - chemin_cru[i].r, chemin_cru[i + 1].c - chemin_cru[i].c));
Console.WriteLine($"Position reelle de l'agent      : {start_vrai}");
Console.WriteLine($"Ce que la colle croit           : {depart_cru} (reidentification tordue)");
Console.WriteLine($"Plan retourne (depuis la croyance) : [{string.Join(", ", chemin_cru)}]  ({moves.Count} mouvements)");

// L'agent execute les mouvements RELATIFS depuis sa position REELLE
var pos = start_vrai;
var trace = new List<(int r, int c)> { pos };
foreach (var mouvement in moves)
{
    pos = (pos.r + mouvement.r, pos.c + mouvement.c);
    trace.Add(pos);
}
Console.WriteLine("Execution pas a pas depuis la position reelle :");
for (int i = 0; i < trace.Count; i++) Console.WriteLine($"   pas {i} : {trace[i]}");
Console.WriteLine($"Arrivee revendiquee : {but} (apres {moves.Count} pas)");
Console.WriteLine($"Position reelle finale : {pos}");

// Verites de terrain (BFS sur la carte entiere)
var map_cells = new HashSet<(int r, int c)>();
for (int r = 0; r < ROWS; r++)
    for (int c = 0; c < COLS; c++)
        if (!WALLS.Contains((r, c))) map_cells.Add((r, c));
var d_full = bfs(map_cells, start_vrai).dist;
int optimum = d_full[but];
Exiger(depart_cru == (3, 6) && chemin_cru.SequenceEqual(new (int, int)[] { (3, 6), (4, 6), (4, 5), (4, 4) }), "Plan cru inattendu");
Exiger(trace.SequenceEqual(new (int, int)[] { (3, 4), (4, 4), (4, 3), (4, 2) }) && pos != but && moves.Count == 3 && optimum == 1 && trace[1] == but, "Témoin exploitable inattendu");
Console.WriteLine($"Optimum reel {start_vrai} -> {but} : {optimum} pas (le but etait le voisin direct)");
Console.WriteLine($"Cout du defaut : arrivee au mauvais endroit ({pos} != {but}), {moves.Count} pas payes");
Console.WriteLine($"contre {optimum} optimal -- et le but a ete TRAVERSE au pas 1 puis quitte.");
Console.WriteLine("Cycle concret exploitant le defaut de recollement : EXECUTE et mesure.");

Position reelle de l'agent      : (3, 4)


Ce que la colle croit           : (3, 6) (reidentification tordue)


Plan retourne (depuis la croyance) : [(3, 6), (4, 6), (4, 5), (4, 4)]  (3 mouvements)


Execution pas a pas depuis la position reelle :


   pas 0 : (3, 4)


   pas 1 : (4, 4)


... (4 outputs left out)

Optimum reel (3, 4) -> (4, 4) : 1 pas (le but etait le voisin direct)


Cout du defaut : arrivee au mauvais endroit ((4, 2) != (4, 4)), 3 pas payes


contre 1 optimal -- et le but a ete TRAVERSE au pas 1 puis quitte.


Cycle concret exploitant le defaut de recollement : EXECUTE et mesure.


### Lecture — le témoin exigé par la Tombée 3

Voilà le **cycle concret qui exploite le défaut** : l'agent traverse son but au premier pas, continue deux pas de plus, et revendique une arrivée à côté. Aucun « résidu », aucune grandeur abstraite : des positions, des pas, un mauvais endroit. C'est la forme pathfinding du Dutch book de de Finetti — la première attestation de l'opération, sur un autre substrat : un objet local plausible, une incohérence invisible par paires, un cycle qui la monnaie. un adversaire qui connaît la table croisée peut router l'agent n'importe où dans le rayon de confusion.

Chiffré pas à pas : le plan retourné depuis la croyance est `[(3, 6), (4, 6), (4, 5), (4, 4)]` (3 mouvements), mais exécuté depuis la position réelle `(3, 4)` il produit les pas `(3, 4) -> (4, 4) -> (4, 3) -> (4, 2)` : l'agent traverse son but `(4, 4)` au pas 1, s'en éloigne, et finit en `(4, 2)` — 3 pas payés contre 1 optimal.

In [16]:
// Reparation : decroiser la table pi_AB, puis tout re-teser
var pi_AB_corrige = new Dictionary<int, int> { [0] = 2, [1] = 3, [2] = 0, [3] = 1 }; // identification cellule a cellule, correcte
show_binding("pi_AB (corrigee)", "A", "B", pi_AB_corrige);
check_pairwise("pi_AB corrigee", pi_AB_corrige);

// sigma recompose
var sigma2 = new Dictionary<(int r, int c), (int r, int c)>();
foreach (var p in PORTALS)
{
    var p_b = convert(p, "A", pi_AB_corrige, "B");
    var p_c = convert(p_b, "B", pi_BC_idx, "C");
    sigma2[p] = convert(p_c, "C", pi_CA_idx, "A");
}
Exiger(PORTALS.All(p => sigma2[p] == p), "La composition corrigée doit être l'identité");
Console.WriteLine("sigma recompose : identite sur tous les portails -- assert OK");

// Le recollement EXISTE desormais : une numerotation globale coherente est exhibee
int n_ok2 = count_global_numberings(pi_AB_corrige, pi_BC_idx, pi_CA_idx);
Console.WriteLine($"Numerotations globales coherentes apres reparation : {n_ok2} / 24");
Exiger(n_ok2 == 24, "Nombre de numérotations corrigées inattendu"); // toute numerotation de depart est coherente : les tables ne contraignent plus

// La table recollee devient univaluee : toute chaine de consultation donne la meme reponse
var paires_portails = new List<((int r, int c) p, (int r, int c) q)>();
for (int i = 0; i < PORTALS.Count; i++)
    for (int j = i + 1; j < PORTALS.Count; j++)
        paires_portails.Add((PORTALS[i], PORTALS[j]));
int paires_testees = 0;
foreach (var paire in paires_portails)
{
    var p = paire.p; var q = paire.q;
    int direct = dists["A"][p][q];
    int via_ab = dists["B"][convert(p, "A", pi_AB_corrige, "B")][convert(q, "A", pi_AB_corrige, "B")];
    int via_ac = dists["C"][convert(convert(p, "A", pi_AB_corrige, "B"), "B", pi_BC_idx, "C")][convert(convert(q, "A", pi_AB_corrige, "B"), "B", pi_BC_idx, "C")];
    Exiger(direct == via_ab && direct == via_ac, $"Réponses divergentes pour {p} -> {q}");
    paires_testees++;
}
Exiger(paires_testees == 6, "Nombre de paires inattendu");
Console.WriteLine($"Univalence : les trois chaines de consultation accordent leurs reponses sur les {paires_testees} paires -- assert OK");

// Exactitude de la table recollee face a la verite de terrain
Console.WriteLine($"{"paire",-22} {"colle (min des fenetres)",24} {"carte entiere",14}");
var distances_paires_citees = new[] { 4, 1, 3, 3, 1, 2 };
for (int i = 0; i < paires_portails.Count; i++)
{
    var paire = paires_portails[i];
    int glued = new[] { "A", "B", "C" }.Min(z => dists[z][paire.p][paire.q]);
    int d_map = bfs(map_cells, paire.p).dist[paire.q];
    string texte_paire = $"({paire.p}, {paire.q})";
    Console.WriteLine($"{texte_paire,-22} {glued,24} {d_map,14}");
    Exiger(glued >= d_map, "Une fenêtre ne crée pas de raccourci impossible"); // une fenetre ne cree jamais de raccourci impossible
    Exiger(glued == distances_paires_citees[i] && d_map == distances_paires_citees[i], "Distance de référence inattendue");
}
Console.WriteLine("La table recollee majore l'optimum global, avec egalite quand une fenetre porte l'optimum : assert OK");

Table pi_AB (corrigee) (A -> B) :


   A#0 (3, 4)  <->  B#2 (3, 4)


   A#1 (4, 4)  <->  B#3 (4, 4)


   A#2 (3, 6)  <->  B#0 (3, 6)


   A#3 (4, 6)  <->  B#1 (4, 6)


pi_AB corrigee : bijection OK, couverture complete OK


... (7 outputs left out)

((3, 6), (4, 4))                              3              3


((3, 6), (4, 6))                              1              1


((4, 4), (4, 6))                              2              2


La table recollee majore l'optimum global, avec egalite quand une fenetre porte l'optimum : assert OK


### Récapitulatif des témoins mesurés

| # | Ce qui est mesuré | Valeur |
|---|---|---|
| Compatibilité par paires | les trois tables passent le contrat faible (bijection, couverture) | ✓ (asserts) |
| Composition sur le triple | σ = π_AB ∘ π_BC ∘ π_CA | **((3,4) (3,6)) ≠ identité** |
| Non-recollabilité | numérotations globales cohérentes (énumération exhaustive) | **0 / 24** |
| Colle multivaluée | même requête physique, deux réponses | **1 contre 3** |
| Témoin exploitable (Loi I) | marche exécutée depuis (3,4) vers (4,4) | **traverse le but au pas un, finit en (4,2)** après trois pas, optimum 1 |
| Réparation | π_AB décroisée → σ = identité, numérotation globale exhibée, univalence des réponses | ✓ (asserts) |

La sortie détaille la réparation : `pi_AB` décroisée associe désormais `A#0 (3, 4)` à `B#2 (3, 4)` (le même portail physique), sigma redevient l'identité sur tous les portails, et l'énumération retrouve 24/24 numérotations cohérentes. L'univalence est vérifiée sur les 6 paires de portails : la colle et la carte entière rendent la même distance à chaque fois (4, 1, 3, 3, 1 et 2 selon la paire).

## Exercices (C.1 — à compléter)

> Les exercices 1 à 3 portent sur LPA\* (sections précédentes) ; les exercices 4 à 6 portent sur la section « recoller ».

Les six exercices reprennent l'instance du carnet (`BASE_WALLS`, `START`, `GOAL`, `blocked`). Chaque stub s'exécute sans erreur une fois complété ; les attendus ont été mesurés avec le code ci-dessus.

Les exercices 4 à 6 suivent la convention du dépôt : stubs sans erreur volontaire, la section s'exécute de bout en bout même non complétée.

In [17]:
// Exercice 1 : reimplanter le coeur du repair -- _update_vertex.
// La classe LPAStarEtu est une copie ou _update_vertex est vide. Completez-la pour que
// le repair retrouve l'optimalite : rhs(u) = min sur les voisins entrants de g(s') + c(s', u),
// puis (re)insertion dans U uniquement si u reste incoherent.
class LPAStarEtu
{
    public void _update_vertex((int x, int y) u)
    {
        // TODO etudiant : recalculer rhs(u) puis maintenir la file U
    }
}
Console.WriteLine("Exercice 1 a completer : repair via _update_vertex etudiant");

Exercice 1 a completer : repair via _update_vertex etudiant


In [18]:
// Exercice 2 : faire croitre la dette du repair nave -- position du blocage.
// Bloquez la cellule plan_init[k] pour k dans {5, 10, 15, 20} et mesurez la dette
// (cout naif - optimal scratch) a chaque fois : ou l'obstacle fait-il le plus mal ?
var dettes = new Dictionary<int, double>();
foreach (int k in new[] { 5, 10, 15, 20 })
{
    // TODO etudiant : bloquer plan_init[k], calculer naif et optimal, stocker la dette
}
Console.WriteLine("Exercice 2 a completer : dette du repair nave selon la position du blocage");
Console.WriteLine($"Obtenu : {{{string.Join(", ", dettes.Select(kv => $"{kv.Key}: {kv.Value}"))}}}");

Exercice 2 a completer : dette du repair nave selon la position du blocage


Obtenu : {}


In [19]:
// Exercice 3 : la clause d'arret. Combien d'expansions le test k_top >= key(goal)
// evite-t-il ? Executez la propagation SANS la clause (propagez TOUTES les incoherences
// restantes de U) puis comparez le nombre d'expansions a celui du repair standard.
int? exp_avec_arret = null; // TODO etudiant : e_repair du carnet (ou re-mesurez-le)
int? exp_sans_arret = null; // TODO etudiant : compter les expansions en vidant U entierement
Console.WriteLine("Exercice 3 a completer : expansions evitees par la clause d'arret");
Console.WriteLine("Attendu : arret des le but coherent (12 expansions) vs vidage complet plus couteux");

Exercice 3 a completer : expansions evitees par la clause d'arret


Attendu : arret des le but coherent (12 expansions) vs vidage complet plus couteux


### Exemple guidé — généraliser la preuve de non-recollabilité

`s=count_global_numberings` enumere les etiquetages injectifs pour trois fenetres et une liste de portails fixee. La generalisation a `k` fenetres et `m` portails ne change que le **graphe des contraintes** : l'enumeration reste `m!` permutations, et chaque table `(z_from, z_to, binding)` ajoute une famille d'egalites a verifier.

L'exemple ci-dessous prend la liste de tables en argument au lieu des trois tables nommees, et rend le compte **et** le nombre d'etiquetages testes.

In [20]:
// Exemple guide -- exercice 4 resolu : k fenetres, m portails
int count_global_numberings_k(
    List<(int r, int c)> portals,
    List<(string z_from, string z_to, Dictionary<int, int> binding)> tables)
{
    // TODO etudiant
    return 0;
}
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 4 — composer deux tables d'intégration

Les tables d'intégration se composent : si `pi_AB` apparie `A#i` à `B#j` et `pi_BC` apparie `B#j` à `C#k`, la table composée `A → C` apparie `A#i` à `C#k`.

Écrivez `compose_bindings(b_ab, b_bc)` qui rend cette table composée, puis vérifiez que le compte de numérotations globales **ne change pas** quand on remplace les deux tables `A → B` et `B → C` par la seule table composée `A → C` — sur les tables défaillantes comme sur les tables réparées.

Indice : composer d'abord, compter ensuite ; l'énumération, elle, ne change pas.

In [21]:
// Exercice 4 -- a completer
Dictionary<int, int> compose_bindings(Dictionary<int, int> b_ab, Dictionary<int, int> b_bc)
{
    // TODO etudiant : composer les deux tables, puis comparer les deux comptes
    return new Dictionary<int, int>();
}
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 5 — un second jeu de portails

Ajoutez le portail `(4,5)` (le centre du bloc `ABC`) et recomposez σ avec la table croisée d'origine : que devient la notation cyclique ? (indice : le nouveau portail est un point fixe de π_AB si la table l'associe à lui-même.)

In [22]:
// Exercice 5 -- a completer
string sigma_avec_portail_central()
{
    // TODO etudiant : reconstruire numbering/by_num avec le portail supplementaire,
    // etendre les tables d'integration (le portail central s'associe a lui-meme), recomposer
    return null; // TODO etudiant
}
var resultat_exercice5 = sigma_avec_portail_central();
Console.WriteLine($"Exercice a completer : {resultat_exercice5 ?? "None"}");

Exercice a completer : None


### Exercice 6 — que fallait-il vérifier par paires ?

In [23]:
// Exercice 6 -- a completer
// Quel test par paires aurait detecte le croisement ICI, et pourquoi il ne suffit pas en general ?
// Reponse attendue (ecrite en commentaire) puis, si vous voulez le mesurer :
// comparerez les profils dists["A"][p][*] et dists["B"][pi_AB_idx(...)][*].
bool? profils_coherents = null; // TODO etudiant : True/False apres mesure des profils A vs B
Console.WriteLine("Exercice a completer");

Exercice a completer


## Mémoire bornée : ce qu'un plafond de frontière coûte

Jusqu'ici, ce carnet comptait le **travail** : les expansions à payer pour réparer un plan. La suite prend l'autre axe, que la recherche informée laisse d'ordinaire en prose — « mémoire limitée », $O(bd)$ — sans jamais le compter : la **mémoire**, et ce qu'il advient de la garantie quand on la borne.

| Question | Ce que la suite compte |
|---|---|
| Combien de mémoire A\* occupe-t-il vraiment ? | le **pic de frontière** (états distincts en attente) **et** les **états atteints** (le dictionnaire jamais vidé) |
| IDA\* tient-il sa promesse de mémoire linéaire ? | le **pic de pile** (longueur du chemin courant) |
| Que fait un plafond de frontière à la garantie d'optimalité ? | le **coût rendu** pour un budget donné, face à l'optimum |

Chaque réponse se mesure sur le même terrain, avec des graines fixées : le taquin 8 et son heuristique de Manhattan, posés dans la section suivante.

## 7. Le terrain : taquin 8, heuristique de Manhattan

L'état est un tuple de neuf entiers, `0` désignant la case vide. Le but est l'ordre naturel. Les instances sont **engendrées depuis le but** par une marche aléatoire à graine fixée : elles sont donc solubles par construction, et reproductibles d'une exécution à l'autre.

La distance de Manhattan est **admissible** (chaque tuile est au moins aussi loin que sa distance en lignes et colonnes) et **consistante** sur ce domaine — c'est ce qui rend les trois algorithmes comparables : aucun ne perd l'optimalité par la faute de l'heuristique.

In [24]:
// Taquin 8 : etat = tuple de 9 entiers, 0 = case vide. Le but est l'ordre naturel.
// Les états sont encodés par une chaîne de neuf chiffres, clé immuable à égalité structurelle.
string BUT = "123456780";
double INFINI = double.PositiveInfinity;
int[][] VOISINS = {
    new[] { 1, 3 }, new[] { 0, 2, 4 }, new[] { 1, 5 },
    new[] { 0, 4, 6 }, new[] { 1, 3, 5, 7 }, new[] { 2, 4, 8 },
    new[] { 3, 7 }, new[] { 4, 6, 8 }, new[] { 5, 7 }
};

int manhattan_taquin(string etat)
{
    int somme = 0;
    for (int i = 0; i < etat.Length; i++)
    {
        int v = etat[i] - '0';
        if (v != 0) somme += Math.Abs(i / 3 - (v - 1) / 3) + Math.Abs(i % 3 - (v - 1) % 3);
    }
    return somme;
}

IEnumerable<string> successeurs(string etat)
{
    int vide = etat.IndexOf('0');
    foreach (int cible in VOISINS[vide])
    {
        var t = etat.ToCharArray();
        (t[vide], t[cible]) = (t[cible], t[vide]);
        yield return new string(t);
    }
}

// CITE : les instances de la marche Python à 20 coups sont reprises comme entrées ci-dessous ;
// elles ne sont pas présentées comme des tirages recalculés par System.Random.

### Des instances reproductibles

Chaque instance est obtenue par une marche aléatoire de 20 coups depuis le but : elle est donc soluble par construction, et la graine fixe l'instance. On mesure d'abord la borne inférieure `h` de chacune. C'est elle qui fixe le coût que l'heuristique annonce — et l'on verra qu'elle ne dit rien de la mémoire que la recherche devra dépenser.

In [25]:
// CITE : données tirées par le jumeau Python pour les graines 1, 2 et 3, reprises telles quelles.
var INSTANCES = new Dictionary<int, string>
{
    [1] = "754801632",
    [2] = "130765824",
    [3] = "813705462"
};
var heuristiques_attendues = new Dictionary<int, int> { [1] = 20, [2] = 10, [3] = 12 };
string afficher_taquin(string etat) => "(" + string.Join(", ", etat.Select(c => c.ToString())) + ")";
Console.WriteLine("instance | h | etat");
Console.WriteLine(new string('-', 46));
foreach (var entree in INSTANCES)
{
    int h = manhattan_taquin(entree.Value);
    Exiger(entree.Value.Length == 9 && entree.Value.OrderBy(c => c).SequenceEqual("012345678") && h == heuristiques_attendues[entree.Key], "Instance ou heuristique inattendue");
    Console.WriteLine($"graine {entree.Key} | h = {h,2} | {afficher_taquin(entree.Value)}");
    for (int ligne = 0; ligne < 3; ligne++)
        Console.WriteLine($"    {afficher_taquin(entree.Value.Substring(ligne * 3, 3))}");
    Console.WriteLine();
}

instance | h | etat


----------------------------------------------


graine 1 | h = 20 | (7, 5, 4, 8, 0, 1, 6, 3, 2)


    (7, 5, 4)


    (8, 0, 1)


    (6, 3, 2)


... (4 outputs left out)

graine 3 | h = 12 | (8, 1, 3, 7, 0, 5, 4, 6, 2)


    (8, 1, 3)


    (7, 0, 5)


    (4, 6, 2)


### Lecture du résultat

Les trois instances n'ont pas la même difficulté, et `h` — la borne inférieure du coût optimal — ne la prédit pas : **20, 10, puis 12**. L'instance de la graine 3 a le `h` le plus faible des trois et c'est pourtant elle qui demandera le plus de mémoire : la section suivante mesure exactement cet écart entre ce que l'heuristique annonce et ce que la frontière coûte.

## 8. A\* : ce que la frontière ne dit pas

A\* en **graphe** garde les états générés non encore dépilés dans sa frontière — mais il garde aussi, jusqu'à la fin, le dictionnaire `g` de **tous** les états atteints. On compte donc trois choses distinctes :

- les **expansions** : les états effectivement finalisés (dépilés une seule fois, avec leur `g` définitif) ;
- le **pic de frontière** : le plus grand nombre d'états distincts simultanément en attente ;
- les **états atteints** : la taille du dictionnaire `g`, qui ne se vide jamais — même frontière vide, même solution trouvée.

La troisième est celle qu'on oublie. Le pic de frontière décrit la **pression d'un moment** ; les états atteints décrivent ce que le processus retient **à la fin**. La mémoire réelle d'un A\* en graphe est la somme des deux, et la mesure ci-dessous montre laquelle domine.

L'`open_set` est tenu à jour à part du tas, parce qu'un tas peut contenir plusieurs entrées périmées pour le même état : compter `tas.Count` mesurerait les doublons, pas la mémoire.

In [26]:
// A* en graphe : coût, expansions, pic de frontière, états atteints et chemin.
(double cout, int expansions, int pic, int etats, List<string> chemin) astar_taquin(string depart)
{
    var g = new Dictionary<string, int> { [depart] = 0 };
    var venu_de = new Dictionary<string, string>();
    var tas = new PriorityQueue<string, (int f, long ordre)>();
    var open_set = new HashSet<string> { depart };
    long compteur = 0;
    tas.Enqueue(depart, (manhattan_taquin(depart), compteur++));
    int expansions = 1, pic = 1;
    while (tas.Count > 0)
    {
        string courant = tas.Dequeue();
        if (!open_set.Contains(courant)) continue; // entree perimee : l'etat a deja ete finalise
        open_set.Remove(courant);
        expansions++;
        if (courant == BUT)
        {
            int cout = g[courant];
            var chemin = new List<string> { courant };
            while (venu_de.TryGetValue(courant, out var precedent))
            {
                courant = precedent;
                chemin.Add(courant);
            }
            chemin.Reverse();
            return (cout, expansions, pic, g.Count, chemin);
        }
        foreach (var suivant in successeurs(courant))
        {
            int ng = g[courant] + 1;
            if (!g.TryGetValue(suivant, out int ancien) || ng < ancien)
            {
                g[suivant] = ng;
                venu_de[suivant] = courant;
                tas.Enqueue(suivant, (ng + manhattan_taquin(suivant), compteur++));
                open_set.Add(suivant);
                pic = Math.Max(pic, open_set.Count);
            }
        }
    }
    return (double.PositiveInfinity, expansions, pic, g.Count, new List<string>());
}
var resultats_astar = INSTANCES.ToDictionary(kv => kv.Key, kv => astar_taquin(kv.Value));
var references_astar = new Dictionary<int, (int cout, int expansions, int pic, int etats)>
{
    [1] = (20, 51, 28, 77), [2] = (14, 51, 39, 88), [3] = (18, 274, 170, 442)
};
Console.WriteLine($"{"graine",6} | {"cout",4} | {"expansions",10} | {"pic frontier",12} | {"etats atteints",14} | {"ratio",5}");
Console.WriteLine(new string('-', 64));
foreach (var kv in resultats_astar)
{
    var r = kv.Value; var attendu = references_astar[kv.Key];
    Exiger(r.cout == attendu.cout && r.expansions == attendu.expansions && r.pic == attendu.pic && r.etats == attendu.etats, "Parité A* taquin");
    Console.WriteLine($"{kv.Key,6} | {r.cout,4} | {r.expansions,10} | {r.pic,12} | {r.etats,14} | {(double)r.etats / r.pic,5:F2}");
}

graine | cout | expansions | pic frontier | etats atteints | ratio


----------------------------------------------------------------


     1 |   20 |         51 |           28 |             77 |  2,75


     2 |   14 |         51 |           39 |             88 |  2,26


     3 |   18 |        274 |          170 |            442 |  2,60


### Lecture du résultat

Les trois instances demandent le **même ordre de grandeur de travail** (51 à 274 expansions) mais des mémoires qui varient d'un facteur **6** : 28, 39, puis **170** états simultanément en attente pour la graine 3. Le pic n'est pas une constante du domaine — c'est une propriété de l'instance.

La dernière colonne corrige une lecture trop charitable du pic : les **états atteints** valent 77, 88 et **442**, soit 2,3 à 2,8 fois le pic de frontière sur les trois instances. Le dictionnaire `g` ne se vide jamais, donc la mémoire d'un A\* en graphe est **dominée par lui**, pas par la frontière. Un carnet qui ne compterait que le pic annoncerait 170 là où le processus en retient 442 : c'est précisément l'erreur que cette colonne existe pour rendre visible.

C'est la raison pour laquelle « A\* tient en mémoire » n'est pas une phrase qu'on peut écrire sans mesurer : sur ce taquin de neuf cases, la frontière d'une instance facile tient dans quelques dizaines d'états, celle d'une instance plus tordue en demande dix fois plus — et le dictionnaire des états atteints, lui, reste en mémoire jusqu'au bout.

Les ratios exacts de la dernière colonne — 2.75, 2.26 et 2.60 — confirment que le dictionnaire des états atteints domine la frontière d'un facteur stable, quelle que soit l'instance.

## 9. IDA\* : la mémoire devient la profondeur

IDA\* n'empile jamais la frontière. Il descend en profondeur sous une borne croissante sur `f = g + h`, et ne garde donc que le **chemin courant**. Sa mémoire est `O(d)` où `d` est la profondeur de la solution, pas le nombre d'états en attente.

Le prix se paie ailleurs : entre deux bornes, les mêmes états sont **revisités**. Le compteur ci-dessous compte les **entrées** dans le parcours en profondeur, toutes itérations confondues — ce n'est pas le même compteur que les expansions d'A\*, qui compte des états finalisés. Les deux ne se comparent donc pas directement ; ce qui se compare, c'est la mémoire.

In [27]:
// IDA* : les entrées comptent les visites sur toutes les bornes successives.
(double cout, int entrees, int pic, List<string> chemin) ida_etoile(string depart)
{
    double borne = manhattan_taquin(depart);
    var chemin = new List<string> { depart };
    int entrees = 0, pic = 1;
    var solution = new List<string>();
    double descendre(int profondeur, double limite)
    {
        string noeud = chemin[chemin.Count - 1];
        double f = profondeur + manhattan_taquin(noeud);
        if (f > limite) return f;
        entrees++;
        pic = Math.Max(pic, chemin.Count);
        if (noeud == BUT)
        {
            solution = new List<string>(chemin);
            return -1;
        }
        double meilleur = double.PositiveInfinity;
        foreach (var suivant in successeurs(noeud))
        {
            if (chemin.Contains(suivant)) continue; // pas de retour en arriere dans le chemin courant
            chemin.Add(suivant);
            double t = descendre(profondeur + 1, limite);
            chemin.RemoveAt(chemin.Count - 1);
            if (t == -1) return -1;
            if (t < meilleur) meilleur = t;
        }
        return meilleur;
    }
    while (true)
    {
        double t = descendre(0, borne);
        if (t == -1) return (solution.Count - 1, entrees, pic, solution);
        if (double.IsPositiveInfinity(t)) return (t, entrees, pic, new List<string>());
        borne = t;
    }
}
var resultats_ida = INSTANCES.ToDictionary(kv => kv.Key, kv => ida_etoile(kv.Value));
var references_ida = new Dictionary<int, (int cout, int entrees, int pic)>
{
    [1] = (20, 21, 21), [2] = (14, 65, 15), [3] = (18, 188, 19)
};
Console.WriteLine($"{"graine",6} | {"cout",4} | {"entrees",8} | {"pic de pile",11}");
Console.WriteLine(new string('-', 36));
foreach (var kv in resultats_ida)
{
    var r = kv.Value; var attendu = references_ida[kv.Key];
    Exiger(r.cout == attendu.cout && r.entrees == attendu.entrees && r.pic == attendu.pic && r.cout == resultats_astar[kv.Key].cout, "Parité IDA*");
    Console.WriteLine($"{kv.Key,6} | {r.cout,4} | {r.entrees,8} | {r.pic,11}");
}

graine | cout |  entrees | pic de pile


------------------------------------


     1 |   20 |       21 |          21


     2 |   14 |       65 |          15


     3 |   18 |      188 |          19


### Lecture du résultat

Le **coût est identique** à celui d'A\* sur les trois instances (20, 14, 18) : IDA\* n'a rien relâché. Le **pic de pile** vaut 21, 15 et 19 — il suit la longueur du chemin, pas la difficulté de l'instance. Sur la graine 3, A\* retenait 442 états atteints pour un pic de frontière de 170, IDA\* en tient 19 : **vingt-trois fois moins** que ce qu'A\* garde, pour le même résultat optimal.

Le compteur d'entrées, lui, n'est pas comparable terme à terme avec les expansions : sur ces instances il tombe entre 21 et 188, tantôt sous les expansions d'A\*, tantôt au-dessus. C'est normal — un état peut être finalisé une fois par A\* et visité plusieurs fois par IDA\* à mesure que la borne monte, et les deux compteurs n'attribuent pas la même chose au même événement. Retenez la **mémoire**, pas le nombre brut.

Sur la graine 2, le contraste est net : 65 entrées dans le parcours pour un pic de pile de 15 — la profondeur borne la mémoire, pas le nombre de visites.

## 10. Le raccourci tentant : plafonner la frontière

Si A\* coûte trop de mémoire, l'idée la plus naturelle est de **plafonner sa frontière** : dès qu'elle dépasse un budget `B`, on jette l'état de plus grand `f`. Le tas devient borné, l'algorithme continue de tourner.

C'est ce que fait `astar_frontiere_bornee` ci-dessous. Trois détails comptent pour la suite :

- le tas peut encore contenir des entrées périmées ; c'est l'`open_set` qui porte le plafond ;
- un état jeté n'est **pas** interdit : s'il est régénéré plus tard avec un `g` meilleur, il revient. Le plafond ne perd donc pas la solution, il perd l'ordre dans lequel A\* la trouvait ;
- le plafond borne la **frontière** seule : `g` et `venu_de` ne sont jamais vidés, donc la mémoire totale mesurée en section 8 ne redescend pas sous le budget. Un plafond de frontière est un plafond de file, pas un plafond de mémoire.

In [28]:
// A* à frontière plafonnée : le pire f est retiré lorsque le budget est dépassé.
// À égalité de f, l'ordre ordinal fixe le départage en C# ; il peut différer de l'ordre d'un set Python.
(double cout, int expansions, int jetes, List<string> chemin) astar_frontiere_bornee(string depart, int budget)
{
    var g = new Dictionary<string, int> { [depart] = 0 };
    var venu_de = new Dictionary<string, string>();
    var tas = new PriorityQueue<string, (int f, long ordre)>();
    var open_set = new HashSet<string> { depart };
    long compteur = 0;
    tas.Enqueue(depart, (manhattan_taquin(depart), compteur++));
    int expansions = 0, jetes = 0;
    while (tas.Count > 0)
    {
        string courant = tas.Dequeue();
        if (!open_set.Remove(courant)) continue;
        expansions++;
        if (courant == BUT)
        {
            int cout = g[courant];
            var chemin = new List<string> { courant };
            while (venu_de.TryGetValue(courant, out var precedent))
            {
                courant = precedent;
                chemin.Add(courant);
            }
            chemin.Reverse();
            return (cout, expansions, jetes, chemin);
        }
        foreach (var suivant in successeurs(courant))
        {
            int ng = g[courant] + 1;
            if (!g.TryGetValue(suivant, out int ancien) || ng < ancien)
            {
                g[suivant] = ng;
                venu_de[suivant] = courant;
                tas.Enqueue(suivant, (ng + manhattan_taquin(suivant), compteur++));
                open_set.Add(suivant);
            }
        }
        while (open_set.Count > budget)
        {
            string victime = open_set.OrderByDescending(e => g[e] + manhattan_taquin(e)).ThenBy(e => e, StringComparer.Ordinal).First();
            open_set.Remove(victime);
            jetes++;
        }
    }
    return (double.PositiveInfinity, expansions, jetes, new List<string>());
}

### Ce que le plafond promet, avant qu'on le mesure

La fonction est déterministe et son contrat est simple : tant que le budget reste au-dessus du pic de frontière de l'instance, elle ne jette **aucun** état et se comporte exactement comme A\* — même coût, même garantie. C'est ce qui autorise un témoin positif : si un budget franc rendait un coût différent de l'optimum, c'est l'instrument qu'il faudrait mettre en cause, pas l'algorithme. Le tableau qui suit chiffre ce point, budget par budget.

In [29]:
// Comparaison des budgets : les références Python sont des valeurs CITÉES.
// À égalité de f, le port retire le plus petit état ordinal ; Python utilise
// le premier maximum dans l'ordre d'itération de son set.
var BUDGETS = new[] { 8, 16, 32, 64, 128 };
var optimum = resultats_astar.ToDictionary(kv => kv.Key, kv => kv.Value.cout);
var references_couts_bornes = new int[,] {
    { 20, 14, 60 }, { 20, 14, 44 }, { 20, 14, 34 },
    { 20, 14, 18 }, { 20, 14, 18 }
};
var references_jetes_bornes = new int[,] {
    { 20, 24, 264 }, { 12, 23, 348 }, { 0, 7, 493 },
    { 0, 0, 99 }, { 0, 0, 42 }
};
const int tolerance_ecart_source = 0;
var mesures_bornees = new Dictionary<(int budget, int graine), (double cout, int expansions, int jetes, List<string> chemin)>();
foreach (int budget in BUDGETS)
    foreach (var instance in INSTANCES.OrderBy(kv => kv.Key))
        mesures_bornees[(budget, instance.Key)] = astar_frontiere_bornee(instance.Value, budget);

Console.WriteLine("Couts calcules par budget et par instance (optimum entre parentheses)");
Console.WriteLine($"{ "budget",6} | {"graine 1 (20)",14} | {"graine 2 (14)",14} | {"graine 3 (18)",14}");
Console.WriteLine(new string('-', 58));
foreach (int budget in BUDGETS)
{
    var valeurs = INSTANCES.Keys.OrderBy(g => g).Select(g =>
    {
        var r = mesures_bornees[(budget, g)];
        return (double.IsPositiveInfinity(r.cout) ? "echec" : r.cout.ToString(CultureInfo.InvariantCulture)).PadLeft(14);
    });
    Console.WriteLine($"{budget,6} | {string.Join(" | ", valeurs)}");
}
Console.WriteLine("\nEtats jetes, cumules sur l'execution");
Console.WriteLine($"{"budget",6} | {"graine 1",10} | {"graine 2",10} | {"graine 3",10}");
Console.WriteLine(new string('-', 44));
foreach (int budget in BUDGETS)
    Console.WriteLine($"{budget,6} | {string.Join(" | ", INSTANCES.Keys.OrderBy(g => g).Select(g => mesures_bornees[(budget, g)].jetes.ToString(CultureInfo.InvariantCulture).PadLeft(10)))}");

Console.WriteLine($"\nComparaison aux references Python CITEES : tolerance {tolerance_ecart_source} ; ecart = port - source.");
for (int i = 0; i < BUDGETS.Length; i++)
{
    int budget = BUDGETS[i];
    foreach (int graine in INSTANCES.Keys.OrderBy(g => g))
    {
        var r = mesures_bornees[(budget, graine)];
        double ecartCout = r.cout - references_couts_bornes[i, graine - 1];
        int ecartJetes = r.jetes - references_jetes_bornes[i, graine - 1];
        bool conforme = Math.Abs(ecartCout) <= tolerance_ecart_source && Math.Abs(ecartJetes) <= tolerance_ecart_source;
        Exiger(r.cout >= optimum[graine], "Un coût rendu ne peut pas améliorer l'optimum");
        if (r.jetes == 0)
            Exiger(r.cout == optimum[graine], "Sans rejet, le coût doit être optimal");
        Console.WriteLine($"budget {budget}, graine {graine} : cout port {r.cout}, source {references_couts_bornes[i, graine - 1]}, ecart {ecartCout:+0;-0;0} ; jetes port {r.jetes}, source {references_jetes_bornes[i, graine - 1]}, ecart {ecartJetes:+0;-0;0} : {(conforme ? "CONFORME" : "ECART : departage de la victime")}");
        if (r.jetes == 0 && references_jetes_bornes[i, graine - 1] == 0)
            Exiger(conforme, "Sans rejet dans les deux exécutions, les résultats doivent coïncider");
    }
}

Couts calcules par budget et par instance (optimum entre parentheses)


budget |  graine 1 (20) |  graine 2 (14) |  graine 3 (18)


----------------------------------------------------------


     8 |             20 |             14 |             60


    16 |             20 |             14 |            132


    32 |             20 |             14 |             22


... (22 outputs left out)

budget 64, graine 3 : cout port 18, source 18, ecart 0 ; jetes port 106, source 99, ecart +7 : ECART : departage de la victime


budget 128, graine 1 : cout port 20, source 20, ecart 0 ; jetes port 0, source 0, ecart 0 : CONFORME


budget 128, graine 2 : cout port 14, source 14, ecart 0 ; jetes port 0, source 0, ecart 0 : CONFORME


budget 128, graine 3 : cout port 18, source 18, ecart 0 ; jetes port 42, source 42, ecart 0 : CONFORME


### Lecture du résultat

Le tableau porte les deux témoins sur la **même instance** :

- **témoin positif** — graine 3, budget 64 : coût **18**, exactement l'optimum d'A\*. Le plafond n'a rien cassé ; il a jeté 99 états sans conséquence sur le résultat.
- **témoin négatif** — graine 3, budget 32 : coût **34**, soit **+89 %** sur l'optimum. À budget 16 : **44** (+144 %). À budget 8 : **60** (+233 %).

Et le détail qui rend le défaut intéressant : sur les graines 1 et 2, même le budget 8 rend l'optimum. Le plafond naïf ne casse pas **toujours** l'optimalité — il la casse quand le budget passe sous ce que l'instance demande, et rien dans l'exécution ne signale qu'on est passé dessous. Une instance facile valide le raccourci ; c'est l'instance difficile qui le dément.

Notez aussi le nombre d'états jetés : 493 à budget 32 sur la graine 3, 99 à budget 64. Un état jeté revient par une autre route — jeter coûte donc du travail en plus du coût en qualité.

La seconde table (« états jetés ») complète le constat : sur la graine 3, 264 états jetés à budget 8, 348 à 16, 493 à 32, puis 99 à 64 et 42 à 128 — et même le budget 128, qui rend l'optimum 18, a jeté 42 états. Sur les graines 1 et 2, le budget 8 rend déjà l'optimum (20 et 14) avec 20 et 24 états jetés.

## 11. Synthèse : trois réponses à « mémoire bornée »

| Stratégie | Mémoire mesurée | Optimalité | Coût réel |
|---|---|---|---|
| **A\*** (graphe) | états atteints : 77 / 88 / **442** (dont un pic de frontière de 28 / 39 / 170) | garantie | référence |
| **IDA\*** | pic de pile : 21 / 15 / **19** | garantie | revisites entre bornes |
| **A\* plafonné** | frontière : **budget fixé** — le dictionnaire `g`, lui, n'est pas borné | perdue sous le seuil | + jetés, + expansions |

Le plafond et IDA\* bornent tous deux la frontière ; seul IDA\* **transporte la garantie**. La différence tient à ce qu'ils oublient : le plafond jette un état sans conserver ce qu'il valait, IDA\* oublie la frontière entière mais la **reconstruit** à chaque borne, ce qui préserve l'ordre d'exploration.

Ne pas confondre les deux colonnes de mémoire : le plafond d'A\* borne la **frontière**, pas le dictionnaire des états atteints — or c'est ce dictionnaire qui domine la mémoire totale mesurée en section 8. Un plafond de frontière ne rend donc pas le processus borné en mémoire.

Entre les deux existe une famille d'algorithmes qui garde les deux propriétés : **SMA\*** (Russell, 1992, *Efficient memory-bounded search methods*) sauvegarde, dans le parent, la plus petite valeur `f` du sous-arbre oublié — l'état jeté redevient un simple candidat à ré-explorer, et l'optimalité survit tant que la solution tient dans le budget. Ce carnet **ne l'implémente pas** : c'est l'exercice 9 qui en pose la question, et une implémentation se mesure avant d'être enseignée.

## Exercice 7 — Mesurer le pic d'une frontière

Écrivez une fonction `pic_de_frontiere(etat)` qui rejoue A\* sur `etat` et rend le **plus grand nombre d'états distincts** simultanément en attente.

Contraintes :

1. ne modifiez pas `astar` : écrivez une variante, ou réutilisez la mesure déjà rendue par `astar` ;
2. le résultat doit être un entier, et non le nombre d'entrées du tas (les doublons périmés ne sont pas de la mémoire) ;
3. vérifiez sur les trois instances : vous devez retrouver 28, 39 et 170.

In [30]:
int pic_de_frontiere(string etat)
{
    // TODO etudiant : rendre le pic de frontiere d'A* sur `etat`.
    // Indice : `astar` rend deja ce pic en troisieme position.
    return 0;
}
// Etape 1 : appelez pic_de_frontiere sur chaque instance et affichez le resultat.
// Etape 2 : comparez a la troisieme colonne du tableau de la section 8.
Console.WriteLine("Exercice 7 a completer");

Exercice 7 a completer


## Exercice 8 — Trouver le budget critique

Pour l'instance de la graine 3, cherchez par **dichotomie** le plus petit budget `B` tel que `astar_frontiere_bornee(etat, B)` rende encore l'optimum 18.

La mesure est falsifiable : le budget trouvé doit rendre 18, et `B - 1` doit rendre strictement plus que 18 (ou échouer). Si votre dichotomie rend un budget qui ne satisfait pas ces deux propriétés, c'est la dichotomie qui est fausse, pas le tableau.

Attention : la propriété « le coût est optimal à partir d'un certain budget » n'est pas démontrée dans ce carnet — elle est **observée** sur cette instance. Une dichotomie suppose cette monotonie ; dites dans votre réponse si vos mesures la confirment.

In [31]:
int budget_critique(string etat, double optimum)
{
    // TODO etudiant : dichotomie sur le budget, entre 1 et le pic de frontiere.
    // Indice : `astar_frontiere_bornee` rend (cout, expansions, jetes, chemin).
    return 0;
}
// Etape 1 : appelez budget_critique sur INSTANCES[3] avec optimum=18.
// Etape 2 : verifiez que le budget rend 18 et que budget - 1 rend autre chose.
Console.WriteLine("Exercice 8 a completer");

Exercice 8 a completer


## Exercice 9 — Un plafond qui ne perd pas la garantie ?

Le plafond naïf jette l'état de plus grand `f` et **oublie ce qu'il valait**. SMA\* propose de s'en souvenir : quand on jette un état, on remonte dans son parent la plus petite valeur `f` du sous-arbre perdu, et le parent redevient un candidat à ré-explorer.

Décrivez, en trois à cinq phrases, **ce que cette sauvegarde change** dans le comportement de l'algorithme, et pourquoi elle suffit à préserver l'optimalité tant que la solution tient en mémoire. Vous pouvez vous appuyer sur le tableau de la section 10 : à budget 32, que faudrait-il que l'algorithme se rappelle pour ne pas rendre 34 ?

Ne codez pas SMA\* ici. Une implémentation fausse enseignée vaut moins qu'une question bien posée.

In [32]:
// TODO etudiant : votre reponse en commentaire, ou dans une chaine affichee.
string reponse_exercice_9 = "";
int verifier_longueur(string texte)
{
    return texte.Replace('!', '.').Replace('?', '.').Split('.').Count(p => !string.IsNullOrWhiteSpace(p));
}
Console.WriteLine($"Exercice 9 a completer : {verifier_longueur(reponse_exercice_9)} phrase(s) detectee(s)");

Exercice 9 a completer : 0 phrase(s) detectee(s)


## Conclusion

Ce carnet constitue une **seconde attestation de l'opération 6**, réparer localement sous garantie, sur le substrat de la recherche de chemin. Trois témoins en précisent le contrat : le repair naïf fournit un chemin valide mais sous-optimal ; LPA\* propage les incohérences et retrouve l'optimum après le même changement ; enfin, au goulet, l'absence de propagation laisse une valeur fausse tandis que la réparation complète retrouve le coût optimal.

La séquence de changements montre l'enchaînement des réparations. Les tables de fenêtres donnent un autre témoin : une composition incorrecte peut produire une marche exploitable malgré des vérifications locales, et sa correction rétablit la cohérence. Le taquin met ensuite en regard travail, mémoire et optimalité pour A\*, IDA\* et une frontière plafonnée.

Dans la série Search, ce carnet prolonge [Search-03 — recherche informée et A\*](Search-03-Informed.ipynb) et [Search-03e — optimalité de A\*](Search-03e-AStar-Optimality.ipynb). Pour approfondir, on pourra étudier D\* Lite lorsque l'agent découvre le monde en se déplaçant, ainsi que les coûts de terrain pondérés. L'[index de la série](README.md) rassemble les autres carnets.

***

## Attestation de parité

Les contrôles exécutables comparent les calculs aux valeurs de référence du carnet Python. Les écarts sont consultables dans les sorties et les assertions des sections suivantes :

| Section | Contrôles |
|---|---|
| Grille et repair | Coûts, longueurs et expansions du calcul initial, du recalcul et des réparations. |
| Séquence et goulet | Concordances, expansions cumulées et croyance après blocage. |
| Recollement | Numérotations, composition, témoin exploitable et distances après réparation. |
| Taquin | Heuristiques, coûts, expansions, pics et états atteints. |
| Budgets | Écarts de coût et d'états jetés par budget et par graine, comparés aux références Python citées avec une tolérance absolue nulle. |